# MACE-MP-0b2 small — DIMER E2E interatomic-potential fine-tuning tutorial (standalone)

[![GitHub](https://img.shields.io/badge/GitHub-181717?style=flat&logo=github&logoColor=white)](https://github.com/kurtvalcorza/mace-materials-pipeline) [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/kurtvalcorza/mace-materials-pipeline/blob/main/tutorials/mace_materials_colab.ipynb) [![Hugging Face](https://img.shields.io/badge/%F0%9F%A4%97%20Hugging%20Face-mace--foundations%2Fmace--mp--0-ffcc4d?style=flat)](https://huggingface.co/mace-foundations/mace-mp-0) [![Upstream](https://img.shields.io/badge/Upstream-ACEsuit%2Fmace-181717?style=flat&logo=github&logoColor=white)](https://github.com/ACEsuit/mace) [![Paper](https://img.shields.io/badge/arXiv-2401.00096-b31b1b.svg)](https://arxiv.org/abs/2401.00096)

**Profile:** `E2E`  
**Mode:** `GUIDED`  
**Notebook specification:** DIMER Notebook Specification 2.0 — **standalone** (§4)  
**Capability:** energy, force and stress prediction for atomic structures and bounded fine-tuning of the foundation interatomic potential

**This notebook is standalone.** It carries the repository's package (3 modules under `src/mace_materials_pipeline/`, at revision `86b95da7c8f7`) verbatim in Section 2, the pinned model identity and the per-file SHA-256 manifest in Section 3, and the exact runtime pins in Section 1, so it keeps working after export even if the repository changes or disappears. Its only external dependencies are the pinned PyPI distributions and the Hugging Face Hub at the immutable revision `e291ace2bfae073c3ebc7ae2f9479a525989baa7` (~68 MB, digest-verified before loading). It was generated by `tools/build_notebook.py` (build_notebook.py/2); edit the repository and regenerate rather than editing cells.

**Run all:** Selecting **Run all** in a fresh supported runtime installs the pinned dependencies (torch, mace-torch, e3nn, ase, numpy, safetensors, huggingface-hub), stages and digest-verifies the pinned MACE-MP-0b2 small source asset (67.6 MB pickled module from the Hub), statically audits every global and generated source string that pickle would execute and refuses anything outside the torch / e3nn / mace allow-lists, unpickles it exactly once to write a code-free JSON config + safetensors pair whose digests are pinned in the module, rebuilds the model from the installed mace-torch and loads it strictly, generates a deterministic 48-structure dataset in code (rattled, strained fcc Cu, Al and Cu₁₆Al₁₆ supercells labelled by ASE's EMT potential — no download), validates the structures and the dataset contract, splits them by composition into train/validation/test sets, checks the frozen model for rotation equivariance, permutation invariance and analytic-versus-finite-difference forces, measures its zero-shot energy and force errors against a composition baseline and a zero-force baseline, calibrates the per-element reference energies and runs a bounded Adam fine-tuning of the readouts plus the last interaction block, evaluates energy and force errors on the held-out test split, predicts three freshly generated structures, exports the adapter as safetensors with a manifest, and reloads that artifact into a fresh pipeline to verify prediction parity. The default path needs no repository clone, no DIMER worker or service, no credential, no upload dialog and no configuration edit (NOTEBOOK_SPEC 2.0 §5). On CPU the whole path takes about two minutes of model time after the download.

**Bring Your Own Data:** After the tutorial workflow completes, set `USE_BYOD = True` in Section 4 and re-run from that cell to supply your own labelled structures as an extended-XYZ file (one frame per structure, a `Lattice` and `pbc` when periodic, an `energy` in eV and a `forces` array in eV/Å per frame). They pass through the same validation, composition-stratified split, baselines, calibration, adaptation, held-out evaluation, inference, artifact export and reload-parity cells as the generated sample. The expected schema, the element support and the ceilings are stated in the Prerequisites and in Section 4, and uploaded files stay inside this runtime. BYOD is optional and never part of the default path.

MACE-MP-0 is a universal machine-learned interatomic potential (Batatia et al., 2023): an equivariant message-passing model trained on Materials Project PBE/PBE+U trajectories that predicts a total energy, per-atom energies, forces and stress for structures of 89 elements. The *0b2 small* checkpoint is the 8.2 M-parameter, two-interaction, 128x0e variant with Agnesi radial transform and ZBL pair repulsion, released November 2024. Forces are the analytic gradient of the energy, so the model needs autograd even at inference — this notebook checks that the forces it returns really are that gradient.

The upstream artifact is a **pickled torch module** (`.model`), which the fleet asset specification treats as executable serialization. So this pipeline never serves it: Section 3 downloads the pinned file, verifies its digest, disassembles it statically (every global it would import, every fx-generated source it would `exec`, every TorchScript source it would compile), refuses anything outside the torch / e3nn / mace allow-lists, unpickles it exactly once, and writes a code-free JSON config + safetensors pair whose digests are pinned in the carried module. The model you run is rebuilt from the installed `mace-torch` and loaded with `strict=True`; the pickle is provenance, not runtime.

The tutorial dataset is generated in code and labelled with **ASE's effective-medium-theory potential (EMT)** — a real, deterministic interatomic potential for Cu and Al, but a different level of theory from the PBE data MACE-MP-0 was trained on. That is exactly the fine-tuning situation: the frozen model already predicts sensible forces, its energies sit on a different reference and its curvature differs, and the adaptation has to close the gap on held-out structures. Every structure of one composition has the same atoms, so a per-element energy regression (the composition baseline) cannot see the displacements at all.

**Learning objectives:** install the pinned runtime; inspect the carried pipeline, dataset and metrics modules; stage and digest-verify an immutable source asset, read a static audit of a pickled checkpoint and see it converted into a code-free serving pair; validate atomic structures and split a labelled dataset by composition; check a foundation potential for equivariance, permutation invariance and gradient-consistent forces; measure zero-shot energy and force errors against two trivial baselines; calibrate reference energies and run a bounded fine-tuning with explicit hyperparameters; evaluate per-atom energy and force errors on an independent test split; predict new structures; and export a safetensors adapter that reloads against the pinned base with verified parity.

**This notebook does not demonstrate:** molecular-dynamics driving, geometry optimisation, phonons, the published MACE-MP-0 benchmarks, the medium and large checkpoints, multi-head replay fine-tuning against the Materials Project data, LAMMPS or CUDA-equivariance deployment, and any claim that an EMT-labelled fcc supercell stands in for a DFT dataset. The repository exposes none of these.

## Prerequisites

- **Runtime:** a fresh supported runtime (Google Colab or Jupyter, Python 3.12). CPU is enough — a 32-atom cell is scored in under 0.1 s and the default fine-tuning takes about a minute — and CUDA is used automatically when present. The model runs in float64, as upstream ships it.
- **Knowledge:** what an interatomic potential, a periodic cell and a force are; why energies are compared per atom; and what MAE and RMSE mean.
- **Executable serialization handled explicitly:** the pinned `.model` file is a pickle. It is digest-verified, statically audited against allow-lists (the audit digest is pinned) and unpickled **once** to produce the safetensors pair the model is actually loaded from. No Hub-hosted Python module is imported; `mace-torch` is installed from PyPI at a pinned version.
- **Data contract:** structures are `{{symbols, positions, cell, pbc}}` in Å with optional `energy` (eV) and `forces` (eV/Å); elements limited to the 89 MACE-MP-0 supports (Z 1–83, 89–94); 1..512 atoms per structure, at most 32 structures and 4,096 atoms per call; no two atoms closer than 0.5 Å (periodic images included); cell vectors under 200 Å and periodic cell heights of at least 1 Å; a dataset needs at least 8 labelled structures with unique names. BYOD accepts extended XYZ.
- **Validation is structural, not chemical:** nothing checks that a structure is charge-neutral, near equilibrium or physically meaningful — a random cloud of supported atoms is scored without complaint, and EMT labels are only meaningful for the metals it parameterises.
- **Privacy:** Do not upload confidential or restricted data to a hosted runtime unless you are authorized to process it there — proprietary alloy compositions or unpublished DFT datasets are exactly that. The default path uploads nothing.
- **External access:** the Hugging Face Hub only, to fetch the pinned `mace-foundations/mace-mp-0` snapshot (~68 MB in total) at revision `e291ace2bfae…`. No GitHub access and no credentials are required; nothing is installed from this repository.

## 1. Install the pinned runtime

The dependency set is pinned exactly (the same `==` pins as the repository's `pyproject.toml` at the generating revision) and installed directly — there is no repository clone and no package install. If a pin replaces a distribution this runtime has already imported, the cell stops with a restart instruction rather than continuing with mixed versions. Look for a dictionary reporting the notebook's source revision, Python, `torch`, `mace`, `e3nn`, `ase` versions, and whether CUDA is available.

In [ ]:
import importlib
import importlib.metadata
import os
import platform
import subprocess
import sys

PINS = [
    'torch==2.14.0',
    'torchvision==0.29.0',
    'mace-torch==0.3.16',
    'e3nn==0.4.4',
    'ase==3.29.0',
    'numpy==2.5.3',
    'safetensors==0.8.0',
    'huggingface-hub==1.32.0',
]
NOTEBOOK_SOURCE = {
    'repository': 'mace-materials-pipeline',
    'repository_revision': '86b95da7c8f7acef68189276d6da12d1907121ef',
    'embedded_module': 'src/mace_materials_pipeline/pipeline.py',
    'embedded_modules': ['src/mace_materials_pipeline/metrics.py', 'src/mace_materials_pipeline/pipeline.py', 'src/mace_materials_pipeline/samples.py'],
    'module_sha256': '35271c8132e6effb3e352aab0891f07e3c3838dd0a0f1f58a018a9c84430a249',
    'generator': 'build_notebook.py/2',
    'notebook_spec': '2.0',
}
SKIP_INSTALL = os.environ.get('DIMER_NOTEBOOK_CI_PREINSTALLED') == '1'

def _installed_version(distribution):
    try:
        return importlib.metadata.version(distribution)
    except importlib.metadata.PackageNotFoundError:
        return None

if not SKIP_INSTALL:
    # Capture every distribution already imported in this runtime, whatever its module name
    # (PIL -> pillow), so a pinned install that replaces a loaded package is detected and the
    # notebook stops with a restart instruction instead of continuing with mixed versions.
    _module_dists = importlib.metadata.packages_distributions()
    _loaded = sorted({d for m in list(sys.modules) for d in _module_dists.get(m.partition('.')[0], ())})
    loaded = {distribution: _installed_version(distribution) for distribution in _loaded}
    subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', *PINS], check=True)
    importlib.invalidate_caches()
    stale = []
    for distribution, before in loaded.items():
        installed = _installed_version(distribution)
        if before is not None and before != installed:
            stale.append(f'{distribution}: loaded={before}, installed={installed}')
    if stale:
        raise RuntimeError('Core dependencies changed while older modules were loaded: ' + '; '.join(stale) + '. Restart the runtime, then rerun from the top.')

import torch, mace, e3nn, ase
print({'notebook_source': NOTEBOOK_SOURCE, 'python': platform.python_version(), 'torch': torch.__version__, 'mace': mace.__version__, 'e3nn': e3nn.__version__, 'ase': ase.__version__, 'cuda': torch.cuda.is_available()})

## 2. Pipeline code (carried verbatim from `src/mace_materials_pipeline/` @ `86b95da7c8f7`)

The next 3 cell(s) **are** the repository's package, module by module in dependency order: the pinned identity constants, snapshot verification (`verify_snapshot`), staged download (`stage_missing_files`), the named operational ceilings, the public validation and evaluation helpers, and the pipeline class. The text is the modules', byte for byte, except for the rewrite rules listed in `tools/build_notebook.py` (1 rule(s), plus the removal of package-relative `from .x import` lines, whose names are already defined by the preceding cells). The repository's parity test (`tests/test_notebook_parity.py`) fails whenever these cells and the modules diverge, so what you run here is what the repository tests. Nothing in these cells runs a model yet.

**Module 1/3:** `src/mace_materials_pipeline/metrics.py`

In [ ]:
"""Energy / force regression metrics and the trivial baselines every adapted number is read against.

Energies are compared per atom (eV/atom) because total energies scale with system size; forces are
compared per Cartesian component (eV/Å). Two baselines need no model: the **composition baseline**
fits one reference energy per element to the training set (the classical "E0" regression, which
cannot see atomic displacements at all) and the **zero-force baseline** predicts every force as zero,
so its MAE is the mean absolute reference force. The third comparison point — the frozen foundation
model after E0 calibration — comes from the pipeline itself.
"""

from __future__ import annotations

import math
from collections.abc import Mapping, Sequence
from typing import Any


def _flatten(rows: Sequence[Sequence[float]]) -> list[float]:
    return [float(x) for row in rows for x in row]


def _mae(errors: Sequence[float]) -> float:
    return sum(abs(e) for e in errors) / len(errors) if errors else math.nan


def _rmse(errors: Sequence[float]) -> float:
    return math.sqrt(sum(e * e for e in errors) / len(errors)) if errors else math.nan


def regression_metrics(references: Sequence[Mapping[str, Any]], predictions: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    """Per-atom energy and per-component force errors between labelled structures and predictions."""
    if len(references) != len(predictions):
        raise ValueError("references and predictions differ in length")
    if not references:
        raise ValueError("at least one structure is required")
    energy_errors: list[float] = []
    force_errors: list[float] = []
    n_atoms = 0
    for ref, pred in zip(references, predictions, strict=True):
        if ref.get("energy") is None or ref.get("forces") is None:
            raise ValueError(f"{ref.get('name', 'structure')}: energy and forces are required")
        count = len(ref["symbols"])
        n_atoms += count
        energy_errors.append((float(pred["energy"]) - float(ref["energy"])) / count)
        pf, rf = _flatten(pred["forces"]), _flatten(ref["forces"])
        if len(pf) != len(rf):
            raise ValueError("force arrays differ in shape")
        force_errors.extend(p - r for p, r in zip(pf, rf, strict=True))
    return {
        "n_structures": len(references),
        "n_atoms": n_atoms,
        "energy_mae_per_atom": _mae(energy_errors),
        "energy_rmse_per_atom": _rmse(energy_errors),
        "force_mae": _mae(force_errors),
        "force_rmse": _rmse(force_errors),
        "units": {"energy": "eV/atom", "forces": "eV/Å"},
    }


def _lstsq(rows: Sequence[Sequence[float]], targets: Sequence[float]) -> list[float]:
    """Least squares via numpy (a light dependency already required for structures)."""
    import numpy as np

    solution, *_ = np.linalg.lstsq(np.array(rows, dtype=float), np.array(targets, dtype=float), rcond=None)
    return [float(x) for x in solution]


def composition_baseline(train: Sequence[Mapping[str, Any]], evaluation: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    """Fit one energy per element on `train` (E = Σ n_element × E0_element) and score `evaluation`.

    It is blind to geometry by construction: every structure of the same composition gets the same
    energy, and its force prediction is identically zero."""
    if not train or not evaluation:
        raise ValueError("train and evaluation sets must be non-empty")
    elements = sorted({s for record in train for s in record["symbols"]})
    rows = [[record["symbols"].count(el) for el in elements] for record in train]
    e0 = _lstsq(rows, [float(record["energy"]) for record in train])
    predictions = []
    for record in evaluation:
        unknown = sorted(set(record["symbols"]) - set(elements))
        if unknown:
            raise ValueError(f"elements {unknown} absent from the training set; the composition baseline cannot score them")
        energy = sum(record["symbols"].count(el) * e for el, e in zip(elements, e0, strict=True))
        predictions.append({"energy": energy, "forces": [[0.0, 0.0, 0.0] for _ in record["symbols"]]})
    metrics = regression_metrics(evaluation, predictions)
    metrics["e0_ev"] = dict(zip(elements, e0, strict=True))
    metrics["baseline"] = "composition (per-element E0 regression, zero forces)"
    return metrics


def zero_force_baseline(evaluation: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    """Force MAE / RMSE of predicting zero force everywhere: the magnitude a model has to beat."""
    if not evaluation:
        raise ValueError("evaluation set must be non-empty")
    components = [x for record in evaluation for x in _flatten(record["forces"])]
    return {
        "n_structures": len(evaluation),
        "force_mae": _mae(components),
        "force_rmse": _rmse(components),
        "units": {"forces": "eV/Å"},
        "baseline": "zero forces",
    }

**Module 2/3:** `src/mace_materials_pipeline/pipeline.py` (carried verbatim; see the note above)

In [ ]:
"""MACE-MP-0b2 small (`mace-foundations/mace-mp-0`) DIMER pipeline: verified snapshot, energy / force /
stress prediction for atomic structures, and bounded fine-tuning of the foundation interatomic potential
to a user's reference data with a portable adapter.

MACE-MP-0 is a universal machine-learned interatomic potential (Batatia et al., 2023): an equivariant
message-passing model trained on the Materials Project PBE/PBE+U trajectories that predicts a total
energy, per-atom energies, forces and stress for any structure of the 89 supported elements. The "0b2
small" checkpoint is the 128x0e, two-interaction, 8.2 M-parameter variant with the Agnesi radial
transform and ZBL pair repulsion, published November 2024.

The upstream distribution format is a **pickled torch module** (`.model`). Under the fleet asset
specification (§11) that is executable serialization, not data, so this package never loads it as a
runtime artifact. Instead:

* `audit_model_file` statically disassembles the pickle (outer archive and the TorchScript archives
  e3nn embeds in it) without executing anything, lists every global it would import and every
  generated source string it would `exec`, and refuses anything outside the torch / e3nn / mace
  allow-lists. The audit result is digest-pinned in `PICKLE_AUDIT_SHA256`.
* `convert_model` unpickles the digest-verified, audit-clean source **once**, extracts the
  constructor arguments and the state dict, and writes a code-free pair: a JSON config plus a
  safetensors file. Their digests are pinned in `CONVERTED_SHA256`; the conversion is deterministic.
* `MaceMaterialsPipeline.from_pretrained` rebuilds `ScaleShiftMACE(**config)` from the **installed**
  `mace-torch` package and loads the safetensors with `strict=True`. The DIMER-hosted artifact is the
  converted pair; the `.model` file is the immutable provenance, not the served asset.

Behaviour is preserved: on three probe structures (rattled Cu and Al supercells, an H2O molecule) the
rebuilt model and the pickled model differ by at most 2.8e-14 eV in energy and 3.6e-15 eV/Å in
forces, and the upstream `MACECalculator` agrees to the same precision (docs/WEIGHTS.md).

Everything model-related is imported lazily so that snapshot verification, the pickle audit and input
validation run (and can refuse) before `torch`, `e3nn` or `mace` are imported (fleet RTM-001). `ase`
and `numpy` are used for structure handling and are imported freely.
"""

from __future__ import annotations

import contextlib
import copy
import hashlib
import io
import json
import math
import pickletools
import re
import time
import warnings
import zipfile
from collections.abc import Callable, Mapping, Sequence
from dataclasses import dataclass, field
from pathlib import Path
from typing import Any


@contextlib.contextmanager
def _float64_default() -> Any:
    """Run a block with torch's default dtype set to float64 (the model's dtype) and hand the caller's default
    back afterwards, even on failure. Loading this pipeline must not change tensor creation elsewhere in the
    hosting process, while every tensor mace-torch builds for us (model, graph data) must be float64."""
    import torch

    previous = torch.get_default_dtype()
    torch.set_default_dtype(torch.float64)
    try:
        yield
    finally:
        torch.set_default_dtype(previous)

MODEL_ID = "mace-foundations/mace-mp-0"
MODEL_REVISION = "e291ace2bfae073c3ebc7ae2f9479a525989baa7"
MODEL_LICENSE = "mit"
MODEL_KEY = "mace-mp-0b2-small"
ARTIFACT_FORMAT = "org.valcorza.mace-materials.adapter.v1"
ARTIFACT_FORMAT_VERSION = "1.0"
ARTIFACT_WEIGHTS_NAME = "adapter.safetensors"
ARTIFACT_MANIFEST_NAME = "manifest.json"
DEFAULT_WEIGHTS_DIR = Path.cwd() / "weights" / MODEL_KEY  # standalone rewrite (build_notebook.py): working-directory-relative
MANIFEST_NAME = "dimer-base-manifest.json"

# Immutable upstream source asset (pickled torch module). Byte-identical to the upstream release asset
# ACEsuit/mace-mp `mace_mp_0b2/mace-small-density-agnesi-stress.model` (2024-11-12); see docs/WEIGHTS.md.
SOURCE_MODEL_NAME = "mace-mp-0b2-small.model"
SOURCE_MODEL_BYTES = 67_622_684
SOURCE_MODEL_SHA256 = "d5773bf9440e96d6eb8c598f84bd0e6369fcfa432f626a87f890e07da3c651c9"
# Code-free serving pair produced deterministically by `convert_model` (asset spec §11.2).
CONVERTED_CONFIG_NAME = "mace-mp-0b2-small.config.json"
CONVERTED_WEIGHTS_NAME = "mace-mp-0b2-small.safetensors"
CONVERTED_SHA256 = {
    CONVERTED_CONFIG_NAME: "130b641179c5da6dbf6ec439b9c20661e3d60512007f6f2d42669d4f87fc9893",
    CONVERTED_WEIGHTS_NAME: "2ed99065c4decf21613b7038dde6bda2b694b2f414012c0ca743f7ff7f86fe93",
}
CONVERTED_BYTES = {CONVERTED_CONFIG_NAME: 3_279, CONVERTED_WEIGHTS_NAME: 67_400_566}
# Digest of the static pickle audit (sorted globals + every generated source string, see audit_model_file).
PICKLE_AUDIT_SHA256 = "9bb150f1ecc9212e594313ba0ef9aa4884b7634dfc8a11da8cbe0c9ebf70d1e5"

# Architecture facts asserted against the converted config before the model library is imported.
MODEL_CLASS = "ScaleShiftMACE"
R_MAX = 5.0
NUM_INTERACTIONS = 2
HIDDEN_IRREPS = "128x0e"
NUM_ELEMENTS = 89
CORRELATION = 3
MAX_ELL = 3
SUPPORTED_ATOMIC_NUMBERS: tuple[int, ...] = tuple(range(1, 84)) + tuple(range(89, 95))  # H..Bi, Ac..Pu
PARAMETER_COUNT = 8_221_984
STATE_TENSORS = 86  # 80 weight / buffer tensors from the pickle + 6 path-pruning flags of mace-torch 0.3.16

# Operational ceilings (structural validation, before any model import).
MAX_STRUCTURES_PER_CALL = 32
MAX_ATOMS_PER_STRUCTURE = 512
MAX_ATOMS_PER_CALL = 4_096
MIN_INTERATOMIC_DISTANCE = 0.5  # Å; closer than any chemistry, the ZBL core would dominate
MAX_CELL_LENGTH = 200.0  # Å
MIN_CELL_HEIGHT = 1.0  # Å for each periodic direction
MAX_ABS_COORDINATE = 10_000.0  # Å
MAX_ABS_ENERGY_PER_ATOM = 1_000.0  # eV
MAX_ABS_FORCE = 1_000.0  # eV/Å

_SYMBOL_RE = re.compile(r"^[A-Z][a-z]?$")

# Static pickle-audit allow-lists (asset spec §11): what a MACE `.model` file may legitimately reference.
PICKLE_ALLOWED_MODULE_ROOTS = ("torch", "e3nn", "mace", "collections", "_codecs", "__torch__")
PICKLE_ALLOWED_EXACT_GLOBALS = frozenset({"__builtin__.set"})
# Every line torch.fx's `reduce_graph_module` would exec from the pickled import block, verbatim.
GENERATED_CODE_ALLOWED_IMPORT_LINES = frozenset(
    {
        "import torch",
        "from math import inf",
        "from math import nan",
        "from torch import device",
        "import torch.fx._pytree as fx_pytree",
        "import torch.utils._pytree as pytree",
        "NoneType = type(None)",
    }
)
GENERATED_CODE_FORBIDDEN = re.compile(
    r"\b(os|subprocess|exec|eval|compile|open|__import__|socket|sys|builtins|importlib|pickle|shutil|"
    r"requests|urllib|globals|locals|breakpoint|input|ctypes|signal|threading|multiprocessing)\b"
)
GENERATED_CODE_CALL_PREFIXES = ("torch.", "fx_pytree.", "pytree.", "math.", "device(")


# --------------------------------------------------------------------------------------------------
# snapshot manifest, staging, static pickle audit and conversion
# --------------------------------------------------------------------------------------------------


def _sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with open(path, "rb") as fh:
        for chunk in iter(lambda: fh.read(1 << 20), b""):
            digest.update(chunk)
    return digest.hexdigest()


def _verify_manifest(root: Path, model_id: str, revision: str) -> dict[str, Any]:
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"no snapshot manifest at {manifest_path}")
    manifest = json.loads(manifest_path.read_text(encoding="utf-8"))
    if manifest.get("modelId") != model_id:
        raise ValueError(f"manifest modelId {manifest.get('modelId')!r} != {model_id!r}")
    if manifest.get("revision") != revision:
        raise ValueError(f"manifest revision {manifest.get('revision')!r} != {revision!r}")
    listed = {entry["path"] for entry in manifest["files"]}
    if SOURCE_MODEL_NAME not in listed:
        raise ValueError(f"manifest does not list {SOURCE_MODEL_NAME}; refusing to proceed")
    for entry in manifest["files"]:
        file_path = root / entry["path"]
        if not file_path.is_file():
            raise FileNotFoundError(f"snapshot file missing: {file_path}")
        size = file_path.stat().st_size
        if size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: size {size} != manifest {entry['bytes']}")
        digest = _sha256_file(file_path)
        if digest != entry["sha256"]:
            raise ValueError(f"{entry['path']}: sha256 {digest} != manifest {entry['sha256']}")
        if entry["path"] == SOURCE_MODEL_NAME and (digest != SOURCE_MODEL_SHA256 or size != SOURCE_MODEL_BYTES):
            raise ValueError(f"{SOURCE_MODEL_NAME}: manifest digest disagrees with the package constant")
    return manifest


def verify_converted(path: str | Path | None = None) -> dict[str, Any]:
    """Check the code-free serving pair (config JSON + safetensors) against the pinned digests."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    report: dict[str, Any] = {"files": []}
    for name, expected in CONVERTED_SHA256.items():
        file_path = root / name
        if not file_path.is_file():
            raise FileNotFoundError(f"converted file missing: {file_path}")
        size = file_path.stat().st_size
        if size != CONVERTED_BYTES[name]:
            raise ValueError(f"{name}: size {size} != pinned {CONVERTED_BYTES[name]}")
        digest = _sha256_file(file_path)
        if digest != expected:
            raise ValueError(f"{name}: sha256 {digest} != pinned {expected}")
        report["files"].append({"path": name, "bytes": size, "sha256": digest})
    return report


def verify_snapshot(path: str | Path | None = None) -> dict[str, Any]:
    """Check the snapshot against its DIMER manifest (size + SHA-256 of every listed Hub file) and,
    when the converted serving pair is present, that pair against the pinned digests."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest = _verify_manifest(root, MODEL_ID, MODEL_REVISION)
    converted = all((root / name).is_file() for name in CONVERTED_SHA256)
    if converted:
        verify_converted(root)
    return {**manifest, "converted": converted}


def _hub_download(relative_path: str, root: Path) -> None:
    """Fetch one manifest-listed file at the pinned revision straight into the snapshot directory."""
    from huggingface_hub import hf_hub_download

    hf_hub_download(MODEL_ID, relative_path, revision=MODEL_REVISION, local_dir=str(root))


def stage_missing_files(
    path: str | Path | None = None,
    *,
    allow_download: bool = False,
    downloader: Callable[[str, Path], None] | None = None,
) -> list[str]:
    """Fetch manifest entries that are absent locally (a fresh clone commits the manifest and the
    3 KB converted config but git-ignores the 67.6 MB `.model` source and the safetensors it converts
    to). `verify_snapshot` still runs after; conversion happens in `from_pretrained`."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    manifest_path = root / MANIFEST_NAME
    if not manifest_path.is_file():
        raise FileNotFoundError(f"manifest not found: {manifest_path}")
    with open(manifest_path, encoding="utf-8") as fh:
        manifest = json.load(fh)
    if manifest.get("modelId") != MODEL_ID or manifest.get("revision") != MODEL_REVISION:
        raise ValueError(
            f"manifest names {manifest.get('modelId')}@{manifest.get('revision')}, "
            f"package pins {MODEL_ID}@{MODEL_REVISION}; refusing to stage"
        )
    missing = [entry["path"] for entry in manifest["files"] if not (root / entry["path"]).is_file()]
    if not missing:
        return []
    if not allow_download:
        raise FileNotFoundError(
            f"snapshot at {root} is missing {missing}; pass allow_download=True to fetch them at {MODEL_REVISION}"
        )
    fetch = downloader or _hub_download
    for relative_path in missing:
        fetch(relative_path, root)
    return missing


def _audit_pickle_bytes(data: bytes, report: dict[str, Any]) -> None:
    """Walk one pickle stream with `pickletools.genops` (no execution) and record what it references."""
    stack: list[Any] = []
    for op, arg, _pos in pickletools.genops(io.BytesIO(data)):
        if op.name == "GLOBAL":  # pickletools renders the (module, name) pair space-separated
            key = arg.replace("\n", " ").replace(" ", ".", 1)
            report["globals"][key] = report["globals"].get(key, 0) + 1
        elif op.name == "STACK_GLOBAL":
            key = f"{stack[-2]}.{stack[-1]}"
            report["globals"][key] = report["globals"].get(key, 0) + 1
        if op.name in ("SHORT_BINUNICODE", "BINUNICODE", "UNICODE", "SHORT_BINSTRING", "BINSTRING"):
            stack.append(arg)
            if isinstance(arg, str):
                if arg.startswith("PK\x03\x04"):
                    report["nested_archives"] += 1
                    _audit_archive_bytes(arg.encode("latin-1"), report)
                elif "def forward" in arg:
                    report["code_strings"].append(arg)
                elif re.search(r"^(?:import|from)\s", arg, re.M):
                    report["import_blocks"].append(arg)
        elif op.name in ("MEMOIZE", "BINPUT", "LONG_BINPUT", "PUT"):
            pass
        else:
            stack.append(None)


def _audit_archive_bytes(blob: bytes, report: dict[str, Any]) -> None:
    archive = zipfile.ZipFile(io.BytesIO(blob))
    for name in archive.namelist():
        if name.endswith(".pkl"):
            _audit_pickle_bytes(archive.read(name), report)
        elif name.endswith(".py"):
            report["torchscript_sources"].append(archive.read(name).decode("utf-8"))


def audit_model_file(path: str | Path | None = None) -> dict[str, Any]:
    """Statically audit the pickled `.model` file without executing it.

    Disassembles the outer torch archive and every nested TorchScript archive with `pickletools`,
    collects every GLOBAL the unpickler would import, every fx-generated source string it would
    `exec`, and every TorchScript source it would compile, then checks them against the allow-lists.
    Raises `ValueError` on any violation. Returns the audit report with its digest."""
    file_path = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR / SOURCE_MODEL_NAME
    if not file_path.is_file():
        raise FileNotFoundError(f"source model not found: {file_path}")
    report: dict[str, Any] = {
        "file": file_path.name,
        "globals": {},
        "nested_archives": 0,
        "code_strings": [],
        "import_blocks": [],
        "torchscript_sources": [],
    }
    _audit_archive_bytes(file_path.read_bytes(), report)
    violations: list[str] = []
    for name in sorted(report["globals"]):
        root = name.split(".")[0]
        if name in PICKLE_ALLOWED_EXACT_GLOBALS or root in PICKLE_ALLOWED_MODULE_ROOTS:
            continue
        violations.append(f"global outside allow-list: {name}")
    import_lines: set[str] = set()
    for block in report["import_blocks"]:
        import_lines.update(line.strip() for line in block.splitlines() if line.strip())
    for code in report["code_strings"]:
        for match in re.finditer(r"^\s*((?:from|import)\s[^\n]+)", code, re.M):
            import_lines.add(match.group(1).strip())
    for line in sorted(import_lines - GENERATED_CODE_ALLOWED_IMPORT_LINES):
        violations.append(f"generated code import line outside allow-list: {line}")
    for kind, sources in (
        ("fx", report["code_strings"]),
        ("import-block", report["import_blocks"]),
        ("torchscript", report["torchscript_sources"]),
    ):
        for src in sources:
            hit = GENERATED_CODE_FORBIDDEN.search(src)
            if hit:
                violations.append(f"forbidden token {hit.group(0)!r} in {kind} source")
                break
    for code in report["code_strings"]:
        for match in re.finditer(r"(?<![\w.])([A-Za-z_][\w.]*)\(", code):
            callee = match.group(1)
            if callee in ("forward", "slice", "int", "float", "tuple", "list", "range", "len", "getattr_1"):
                continue
            if not callee.startswith(GENERATED_CODE_CALL_PREFIXES) and "." not in callee:
                violations.append(f"generated code calls a bare name: {callee}")
                break
    summary = {
        "globals": sorted(report["globals"]),
        "nested_archives": report["nested_archives"],
        "fx_code_strings": len(report["code_strings"]),
        "import_blocks": len(report["import_blocks"]),
        "torchscript_sources": len(report["torchscript_sources"]),
        "generated_code_import_lines": sorted(import_lines),
        "violations": violations,
    }
    digest = hashlib.sha256()
    digest.update("\n".join(summary["globals"]).encode("utf-8"))
    digest.update(b"\n--fx--\n" + "\n---\n".join(sorted(report["code_strings"])).encode("utf-8"))
    digest.update(b"\n--imports--\n" + "\n---\n".join(sorted(report["import_blocks"])).encode("utf-8"))
    digest.update(b"\n--ts--\n" + "\n---\n".join(sorted(report["torchscript_sources"])).encode("utf-8"))
    summary["audit_sha256"] = digest.hexdigest()
    if violations:
        raise ValueError(f"{file_path.name}: pickle audit failed: {violations}")
    return summary


def _config_from_json(config: Mapping[str, Any]) -> dict[str, Any]:
    """Assert the converted config names the pinned architecture (before any model import)."""
    expected = {
        "model_class": MODEL_CLASS,
        "r_max": R_MAX,
        "num_interactions": NUM_INTERACTIONS,
        "hidden_irreps": HIDDEN_IRREPS,
        "num_elements": NUM_ELEMENTS,
        "correlation": CORRELATION,
        "max_ell": MAX_ELL,
        "dtype": "float64",
        "heads": ["default"],
    }
    for key, value in expected.items():
        if config.get(key) != value:
            raise ValueError(f"converted config {key}={config.get(key)!r} disagrees with the package constant {value!r}")
    if tuple(config.get("atomic_numbers", ())) != SUPPORTED_ATOMIC_NUMBERS:
        raise ValueError("converted config atomic_numbers disagree with SUPPORTED_ATOMIC_NUMBERS")
    if len(config.get("atomic_energies", ())) != NUM_ELEMENTS:
        raise ValueError("converted config atomic_energies length != NUM_ELEMENTS")
    return dict(config)


def _json_ready(value: Any) -> Any:
    """Serialise `extract_config_mace_model` output: classes and callables by name, irreps as strings."""
    import numpy as np
    import torch
    from e3nn import o3

    if isinstance(value, type):
        return value.__name__
    if isinstance(value, o3.Irreps):
        return str(value)
    if callable(value):
        return value.__name__
    if isinstance(value, (np.ndarray, torch.Tensor)):
        return value.tolist()
    if isinstance(value, (list, tuple)):
        return [int(x) if isinstance(x, np.integer) else x for x in value]
    return value


def build_model(config: Mapping[str, Any]) -> Any:
    """Instantiate `ScaleShiftMACE` from the converted JSON config using the installed mace-torch."""
    import mace  # noqa: F401  (import order: mace sets TORCH_FORCE_NO_WEIGHTS_ONLY_LOAD before e3nn loads its constants)
    import numpy as np
    import torch
    from e3nn import o3
    from mace.modules import ScaleShiftMACE, blocks

    # The model is float64: construct it under that default and hand the caller's default back afterwards.
    with _float64_default():
        kwargs = {k: v for k, v in config.items() if k not in ("model_class", "dtype")}
        kwargs["interaction_cls"] = getattr(blocks, kwargs["interaction_cls"])
        kwargs["interaction_cls_first"] = getattr(blocks, kwargs["interaction_cls_first"])
        kwargs["readout_cls"] = getattr(blocks, kwargs["readout_cls"])
        kwargs["hidden_irreps"] = o3.Irreps(kwargs["hidden_irreps"])
        kwargs["MLP_irreps"] = o3.Irreps(kwargs["MLP_irreps"])
        kwargs["gate"] = {"silu": torch.nn.functional.silu}[kwargs["gate"]]
        kwargs["atomic_energies"] = np.array(kwargs["atomic_energies"], dtype=float)
        kwargs["atomic_inter_scale"] = float(kwargs["atomic_inter_scale"])
        kwargs["atomic_inter_shift"] = float(kwargs["atomic_inter_shift"])
        if kwargs.get("edge_irreps") is not None:
            kwargs["edge_irreps"] = o3.Irreps(kwargs["edge_irreps"])
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            return ScaleShiftMACE(**kwargs)


# State-dict keys of the pickled checkpoint that mace-torch 0.3.16 no longer registers (the ZBL block now
# derives its cutoff from covalent radii); their values (p=5, r_max=5.0) are not used by the current forward.
STALE_SOURCE_KEYS = ("pair_repulsion_fn.cutoff.p", "pair_repulsion_fn.cutoff.r_max", "pair_repulsion_fn.r_max")


def convert_model(path: str | Path | None = None, *, audit: bool = True) -> dict[str, Any]:
    """Convert the digest-verified `.model` pickle into the code-free serving pair, deterministically.

    Order of operations: digest check → static audit (refuses on any allow-list violation or audit
    digest drift) → single unpickle (`torch.load(weights_only=False)`, the only place this package
    executes anything from the file) → `extract_config_mace_model` → JSON config + safetensors state
    dict of a freshly constructed model that loaded the pickled tensors. Returns the digests."""
    root = Path(path) if path is not None else DEFAULT_WEIGHTS_DIR
    source = root / SOURCE_MODEL_NAME
    if not source.is_file():
        raise FileNotFoundError(f"source model not found: {source}")
    size = source.stat().st_size
    if size != SOURCE_MODEL_BYTES:
        raise ValueError(f"{SOURCE_MODEL_NAME}: size {size} != pinned {SOURCE_MODEL_BYTES}")
    digest = _sha256_file(source)
    if digest != SOURCE_MODEL_SHA256:
        raise ValueError(f"{SOURCE_MODEL_NAME}: sha256 {digest} != pinned {SOURCE_MODEL_SHA256}")
    if audit:
        summary = audit_model_file(source)
        if summary["audit_sha256"] != PICKLE_AUDIT_SHA256:
            raise ValueError(
                f"{SOURCE_MODEL_NAME}: pickle audit digest {summary['audit_sha256']} != pinned {PICKLE_AUDIT_SHA256}"
            )
    import mace  # noqa: F401
    import torch
    from mace.tools.scripts_utils import extract_config_mace_model
    from safetensors.torch import save_file

    # Conversion runs under a float64 default and restores the caller's default even on failure.
    with _float64_default():
        started = time.perf_counter()
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            pickled = torch.load(source, map_location="cpu", weights_only=False)
        if type(pickled).__name__ != MODEL_CLASS:
            raise ValueError(f"{SOURCE_MODEL_NAME} unpickled to {type(pickled).__name__}, expected {MODEL_CLASS}")
        config = {k: _json_ready(v) for k, v in extract_config_mace_model(pickled).items()}
        config["model_class"] = type(pickled).__name__
        config["dtype"] = str(next(pickled.parameters()).dtype).replace("torch.", "")
        _config_from_json(config)
        text = json.dumps(config, indent=2, sort_keys=True) + "\n"
        (root / CONVERTED_CONFIG_NAME).write_bytes(text.encode("utf-8"))

        source_state = {k: v.contiguous() for k, v in pickled.state_dict().items()}
        model = build_model(config)
        result = model.load_state_dict({k: v for k, v in source_state.items() if k not in STALE_SOURCE_KEYS}, strict=False)
        if result.unexpected_keys or any(not k.endswith("_zeroed") for k in result.missing_keys):
            raise ValueError(
                f"state-dict layout drift between the pickle and mace-torch: "
                f"missing={result.missing_keys} unexpected={result.unexpected_keys}"
            )
        canonical = {k: v.contiguous() for k, v in model.state_dict().items()}
        if len(canonical) != STATE_TENSORS:
            raise ValueError(f"converted state dict has {len(canonical)} tensors, expected {STATE_TENSORS}")
        for key, tensor in source_state.items():
            if key not in STALE_SOURCE_KEYS and not torch.equal(tensor, canonical[key]):
                raise ValueError(f"converted tensor {key} is not bit-identical to the pickled one")
        save_file(canonical, str(root / CONVERTED_WEIGHTS_NAME), metadata={"format": "pt"})
        report = verify_converted(root)
        return {
            "source": {"path": SOURCE_MODEL_NAME, "bytes": size, "sha256": digest},
            "converted": report["files"],
            "dropped_keys": list(STALE_SOURCE_KEYS),
            "added_flag_keys": sorted(result.missing_keys),
            "seconds": round(time.perf_counter() - started, 2),
        }


# --------------------------------------------------------------------------------------------------
# structures and validation (no model import)
# --------------------------------------------------------------------------------------------------

INPUT_SCHEMA: dict[str, Any] = {
    "input": (
        "1..MAX_STRUCTURES_PER_CALL structures, each {symbols, positions, cell, pbc} (Å), "
        "optionally energy (eV) and forces (eV/Å)"
    ),
    "structures": [1, MAX_STRUCTURES_PER_CALL],
    "atoms_per_structure": [1, MAX_ATOMS_PER_STRUCTURE],
    "atoms_per_call": [1, MAX_ATOMS_PER_CALL],
    "elements": "the 89 elements of MACE-MP-0 (Z = 1..83 and 89..94)",
    "min_interatomic_distance_angstrom": MIN_INTERATOMIC_DISTANCE,
    "cell": f"3x3 lattice vectors in Å, each shorter than {MAX_CELL_LENGTH} Å; a cell is required whenever any pbc flag is true",
    "validation": (
        "element support, finite coordinates, cell/pbc consistency and interatomic-distance and size ceilings only. "
        "Nothing checks that a structure is chemically sensible, charge-neutral or near equilibrium; a random cloud "
        "of supported atoms is scored without complaint"
    ),
    "units": "energies in eV, forces in eV/Å, stress in eV/Å³ (ASE conventions)",
}


def _atomic_symbols() -> dict[str, int]:
    from ase.data import atomic_numbers

    return dict(atomic_numbers)


def _check_structure(structure: Mapping[str, Any], index: int) -> dict[str, Any]:
    """Return a normalised copy of one structure or raise ValueError describing the first defect."""
    import numpy as np

    label = f"structure[{index}]"
    if not isinstance(structure, Mapping):
        raise ValueError(f"{label} must be a mapping with symbols/positions/cell/pbc, got {type(structure).__name__}")
    symbols = structure.get("symbols")
    positions = structure.get("positions")
    if not isinstance(symbols, (list, tuple)) or not symbols:
        raise ValueError(f"{label}: symbols must be a non-empty list of element symbols")
    n_atoms = len(symbols)
    if n_atoms > MAX_ATOMS_PER_STRUCTURE:
        raise ValueError(f"{label}: {n_atoms} atoms exceed the ceiling of {MAX_ATOMS_PER_STRUCTURE}")
    table = _atomic_symbols()
    numbers = []
    for symbol in symbols:
        if not isinstance(symbol, str) or not _SYMBOL_RE.match(symbol) or symbol not in table:
            raise ValueError(f"{label}: {symbol!r} is not an element symbol")
        z = table[symbol]
        if z not in SUPPORTED_ATOMIC_NUMBERS:
            raise ValueError(f"{label}: element {symbol} (Z={z}) is outside the 89 elements MACE-MP-0 supports")
        numbers.append(z)
    try:
        pos = np.asarray(positions, dtype=float)
    except (TypeError, ValueError) as exc:
        raise ValueError(f"{label}: positions must be an (n_atoms, 3) array of numbers") from exc
    if pos.shape != (n_atoms, 3):
        raise ValueError(f"{label}: positions shape {pos.shape} != ({n_atoms}, 3)")
    if not np.all(np.isfinite(pos)) or np.abs(pos).max() > MAX_ABS_COORDINATE:
        raise ValueError(f"{label}: positions must be finite and within ±{MAX_ABS_COORDINATE} Å")
    pbc_raw = structure.get("pbc", False)
    if isinstance(pbc_raw, bool):
        pbc = [pbc_raw] * 3
    else:
        pbc = list(pbc_raw) if isinstance(pbc_raw, (list, tuple)) else None
        if pbc is None or len(pbc) != 3 or not all(isinstance(flag, (bool, int)) for flag in pbc):
            raise ValueError(f"{label}: pbc must be a bool or a list of three bools")
        pbc = [bool(flag) for flag in pbc]
    cell_raw = structure.get("cell")
    cell = None
    if cell_raw is not None:
        try:
            cell = np.asarray(cell_raw, dtype=float)
        except (TypeError, ValueError) as exc:
            raise ValueError(f"{label}: cell must be a 3x3 array of numbers") from exc
        if cell.shape != (3, 3) or not np.all(np.isfinite(cell)):
            raise ValueError(f"{label}: cell must be a finite 3x3 array")
        lengths = np.linalg.norm(cell, axis=1)
        if lengths.max() > MAX_CELL_LENGTH:
            raise ValueError(f"{label}: cell vector of {lengths.max():.1f} Å exceeds {MAX_CELL_LENGTH} Å")
        volume = abs(float(np.linalg.det(cell)))
        if any(pbc):
            if volume <= 0.0:
                raise ValueError(f"{label}: periodic cell is singular")
            for axis in range(3):
                if pbc[axis]:
                    cross = np.cross(cell[(axis + 1) % 3], cell[(axis + 2) % 3])
                    height = volume / np.linalg.norm(cross)
                    if height < MIN_CELL_HEIGHT:
                        raise ValueError(
                            f"{label}: periodic cell height {height:.3f} Å along axis {axis} is below {MIN_CELL_HEIGHT} Å"
                        )
    elif any(pbc):
        raise ValueError(f"{label}: pbc requests periodicity but no cell is given")
    from ase.geometry import get_distances

    if n_atoms > 1:
        _, dists = get_distances(pos, cell=cell if any(pbc) else None, pbc=pbc if any(pbc) else None)
        np.fill_diagonal(dists, np.inf)
        d_min = float(dists.min())
        if d_min < MIN_INTERATOMIC_DISTANCE:
            raise ValueError(f"{label}: two atoms are {d_min:.3f} Å apart, below the {MIN_INTERATOMIC_DISTANCE} Å floor")
    else:
        d_min = math.inf
    energy = structure.get("energy")
    forces_raw = structure.get("forces")
    if energy is not None:
        if isinstance(energy, bool) or not isinstance(energy, (int, float)) or not math.isfinite(float(energy)):
            raise ValueError(f"{label}: energy must be a finite number in eV")
        if abs(float(energy)) / n_atoms > MAX_ABS_ENERGY_PER_ATOM:
            raise ValueError(f"{label}: |energy| per atom exceeds {MAX_ABS_ENERGY_PER_ATOM} eV")
    forces = None
    if forces_raw is not None:
        try:
            forces = np.asarray(forces_raw, dtype=float)
        except (TypeError, ValueError) as exc:
            raise ValueError(f"{label}: forces must be an (n_atoms, 3) array of numbers") from exc
        if forces.shape != (n_atoms, 3) or not np.all(np.isfinite(forces)) or np.abs(forces).max() > MAX_ABS_FORCE:
            raise ValueError(f"{label}: forces must be a finite ({n_atoms}, 3) array within ±{MAX_ABS_FORCE} eV/Å")
    name = structure.get("name")
    if name is not None and (not isinstance(name, str) or len(name) > 200):
        raise ValueError(f"{label}: name must be a string of at most 200 characters")
    return {
        "name": name if name is not None else f"structure-{index}",
        "symbols": [str(s) for s in symbols],
        "numbers": numbers,
        "positions": pos.tolist(),
        "cell": cell.tolist() if cell is not None else None,
        "pbc": pbc,
        "energy": float(energy) if energy is not None else None,
        "forces": forces.tolist() if forces is not None else None,
        "n_atoms": n_atoms,
        "min_distance": d_min,
        "periodic": any(pbc),
    }


def _check_structures(structures: Any) -> list[dict[str, Any]]:
    if isinstance(structures, Mapping) or not isinstance(structures, Sequence) or isinstance(structures, (str, bytes)):
        raise ValueError("structures must be a list of structure mappings")
    if not structures:
        raise ValueError("at least one structure is required")
    if len(structures) > MAX_STRUCTURES_PER_CALL:
        raise ValueError(f"{len(structures)} structures exceed the ceiling of {MAX_STRUCTURES_PER_CALL} per call")
    checked = [_check_structure(s, i) for i, s in enumerate(structures)]
    total = sum(s["n_atoms"] for s in checked)
    if total > MAX_ATOMS_PER_CALL:
        raise ValueError(f"{total} atoms in one call exceed the ceiling of {MAX_ATOMS_PER_CALL}")
    return checked


def structure_digest(structure: Mapping[str, Any]) -> str:
    """SHA-256 of the geometry (symbols, positions rounded to 1e-6 Å, cell, pbc); labels excluded."""
    checked = _check_structure(structure, 0)
    payload = {
        "symbols": checked["symbols"],
        "positions": [[round(x, 6) for x in row] for row in checked["positions"]],
        "cell": [[round(x, 6) for x in row] for row in checked["cell"]] if checked["cell"] else None,
        "pbc": checked["pbc"],
    }
    return hashlib.sha256(json.dumps(payload, separators=(",", ":")).encode("utf-8")).hexdigest()


def validate_inputs(structures: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
    """Structural validation only; raises ValueError before any model library is imported."""
    checked = _check_structures(structures)
    return {
        "n_structures": len(checked),
        "n_atoms": sum(s["n_atoms"] for s in checked),
        "elements": sorted({s for c in checked for s in c["symbols"]}),
        "periodic": [c["periodic"] for c in checked],
        "min_distance": min(c["min_distance"] for c in checked),
        "labelled": sum(1 for c in checked if c["energy"] is not None and c["forces"] is not None),
    }


def to_atoms(structure: Mapping[str, Any]) -> Any:
    """ase.Atoms for one validated structure (labels attached as `info['energy']` / `arrays['forces']`)."""
    import numpy as np
    from ase import Atoms

    checked = _check_structure(structure, 0)
    atoms = Atoms(symbols=checked["symbols"], positions=np.array(checked["positions"]), pbc=checked["pbc"])
    if checked["cell"] is not None:
        atoms.set_cell(np.array(checked["cell"]))
    if checked["energy"] is not None:
        atoms.info["energy"] = checked["energy"]
    if checked["forces"] is not None:
        atoms.arrays["forces"] = np.array(checked["forces"])
    atoms.info["name"] = checked["name"]
    return atoms


def from_atoms(atoms: Any, *, name: str | None = None) -> dict[str, Any]:
    """Structure mapping from ase.Atoms; picks up energy/forces from `info`/`arrays` or a SinglePointCalculator."""
    import numpy as np

    energy = atoms.info.get("energy")
    forces = atoms.arrays.get("forces")
    calc = getattr(atoms, "calc", None)
    if calc is not None and getattr(calc, "results", None):
        energy = calc.results.get("energy", energy)
        forces = calc.results.get("forces", forces)
    cell = np.array(atoms.get_cell())
    has_cell = bool(np.any(cell != 0.0))
    return {
        "name": name or atoms.info.get("name"),
        "symbols": list(atoms.get_chemical_symbols()),
        "positions": atoms.get_positions().tolist(),
        "cell": cell.tolist() if has_cell else None,
        "pbc": [bool(x) for x in atoms.get_pbc()],
        "energy": float(energy) if energy is not None else None,
        "forces": np.asarray(forces, dtype=float).tolist() if forces is not None else None,
    }


# --------------------------------------------------------------------------------------------------
# pipeline
# --------------------------------------------------------------------------------------------------

TRAINABLE_ALWAYS = ("readouts.", "atomic_energies_fn.", "scale_shift.")
ENERGY_WEIGHT = 100.0  # on the per-atom energy MSE (eV/atom)²
FORCES_WEIGHT = 10.0  # on the force-component MSE (eV/Å)²


def _trainable_prefixes(trainable_blocks: int) -> tuple[str, ...]:
    if not isinstance(trainable_blocks, int) or not 0 <= trainable_blocks <= NUM_INTERACTIONS:
        raise ValueError(f"trainable_blocks must be an int in 0..{NUM_INTERACTIONS}")
    prefixes = list(TRAINABLE_ALWAYS)
    for k in range(NUM_INTERACTIONS - trainable_blocks, NUM_INTERACTIONS):
        prefixes += [f"interactions.{k}.", f"products.{k}."]
    return tuple(prefixes)


@dataclass
class MaceMaterialsPipeline:
    """Energy / force / stress prediction and bounded fine-tuning on top of the verified MACE-MP-0b2 small model."""

    model: Any
    config: dict[str, Any]
    device: str
    weights_dir: Path
    source: str
    adapter: dict[str, Any] | None = None
    _z_table: Any = field(default=None, repr=False)

    @classmethod
    def from_pretrained(
        cls,
        *,
        device: str = "cpu",
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
        require_source: bool = True,
        report: Callable[[dict[str, Any]], None] | None = None,
    ) -> MaceMaterialsPipeline:
        """Verify, convert if needed, rebuild and strictly load. With `require_source=False` the pickled
        source may be absent (the DIMER-hosted case) as long as the converted pair verifies. `report`
        receives the static-audit summary and the conversion record when a conversion happens."""
        root = Path(weights_dir) if weights_dir is not None else DEFAULT_WEIGHTS_DIR
        if require_source:
            stage_missing_files(root, allow_download=allow_download)
            snapshot = verify_snapshot(root)
            if not snapshot["converted"]:
                if report is not None:
                    audit = audit_model_file(root / SOURCE_MODEL_NAME)
                    report(
                        {
                            "pickle_audit": {k: v for k, v in audit.items() if k != "globals"},
                            "global_count": len(audit["globals"]),
                        }
                    )
                conversion = convert_model(root)
                if report is not None:
                    report({"conversion": conversion})
                snapshot = verify_snapshot(root)
            elif report is not None:
                report({"conversion": "converted pair already present and digest-verified"})
            source = "converted from the manifest-verified source pickle"
        else:
            # Converted-only deployment: only the code-free pair is verified, whether or not the committed
            # source manifest happens to sit beside it; the pickle is never required or fetched here.
            verify_converted(root)
            source = "converted pair, pinned digests (source pickle not required)"
        config = _config_from_json(json.loads((root / CONVERTED_CONFIG_NAME).read_text(encoding="utf-8")))
        import torch
        from safetensors.torch import load_file

        model = build_model(config)
        state = load_file(str(root / CONVERTED_WEIGHTS_NAME))
        model.load_state_dict(state, strict=True)
        n_params = sum(p.numel() for p in model.parameters())
        if n_params != PARAMETER_COUNT:
            raise ValueError(f"rebuilt model has {n_params} parameters, expected {PARAMETER_COUNT}")
        model.to(torch.device(device)).eval()
        for p in model.parameters():
            p.requires_grad_(False)
        from mace.tools import AtomicNumberTable

        return cls(
            model=model,
            config=config,
            device=device,
            weights_dir=root,
            source=source,
            _z_table=AtomicNumberTable(list(SUPPORTED_ATOMIC_NUMBERS)),
        )

    # ---- batching -------------------------------------------------------------------------------------

    def _dataset(self, checked: Sequence[Mapping[str, Any]], *, labels: bool) -> list[Any]:
        from mace.data import AtomicData, config_from_atoms
        from mace.data.utils import KeySpecification

        spec = KeySpecification(info_keys={"energy": "energy"}, arrays_keys={"forces": "forces"})
        data = []
        with _float64_default():  # graph tensors follow the default dtype; the model is float64
            for structure in checked:
                atoms = to_atoms(structure)
                if labels and (structure["energy"] is None or structure["forces"] is None):
                    raise ValueError(f"{structure['name']}: energy and forces are required for this operation")
                conf = config_from_atoms(atoms, key_specification=spec)
                data.append(AtomicData.from_config(conf, z_table=self._z_table, cutoff=R_MAX))
        return data

    def _loader(self, data: Sequence[Any], batch_size: int, shuffle: bool, seed: int = 0) -> Any:
        import torch
        from mace.tools import torch_geometric

        generator = torch.Generator().manual_seed(seed) if shuffle else None
        return torch_geometric.dataloader.DataLoader(
            list(data), batch_size=batch_size, shuffle=shuffle, drop_last=False, generator=generator
        )

    def _forward(self, model: Any, batch: Any, *, training: bool, stress: bool) -> dict[str, Any]:
        import torch

        data = batch.to_dict()
        # Forces are -dE/dx, so the forward needs autograd even at inference; no_grad would zero them.
        out = model(data, training=training, compute_force=True, compute_stress=stress)
        if not training:
            out = {k: (v.detach() if isinstance(v, torch.Tensor) else v) for k, v in out.items()}
        return out

    # ---- inference ------------------------------------------------------------------------------------

    def predict(self, structures: Sequence[Mapping[str, Any]], *, batch_size: int = 8) -> dict[str, Any]:
        """Total energy (eV), per-atom energies, forces (eV/Å) and, for fully periodic structures, stress (eV/Å³)."""
        checked = _check_structures(structures)
        started = time.perf_counter()
        data = self._dataset(checked, labels=False)
        model = self.model
        # Stress is a batch-level switch in the MACE forward, so fully periodic structures (stress-eligible)
        # and everything else are batched separately; results are put back in the caller's order.
        eligible = [i for i, s in enumerate(checked) if s["periodic"] and all(s["pbc"])]
        others = [i for i in range(len(checked)) if i not in set(eligible)]
        results: list[dict[str, Any] | None] = [None] * len(checked)
        for indices, stress in ((eligible, True), (others, False)):
            if not indices:
                continue
            position = 0
            for batch in self._loader([data[i] for i in indices], batch_size, shuffle=False):
                out = self._forward(model, batch, training=False, stress=stress)
                ptr = batch.ptr.tolist()
                energies = out["energy"].cpu().tolist()
                node_energy = out["node_energy"].cpu()
                forces = out["forces"].cpu()
                stresses = out.get("stress")
                for i in range(len(ptr) - 1):
                    lo, hi = ptr[i], ptr[i + 1]
                    results[indices[position]] = {
                        "energy": float(energies[i]),
                        "energy_per_atom": float(energies[i]) / (hi - lo),
                        "node_energies": node_energy[lo:hi].tolist(),
                        "forces": forces[lo:hi].tolist(),
                        "stress": stresses[i].cpu().tolist() if (stress and stresses is not None) else None,
                    }
                    position += 1
        if any(r is None for r in results):
            raise RuntimeError("prediction did not cover every structure")
        for structure, result in zip(checked, results, strict=True):
            result["name"] = structure["name"]
            result["n_atoms"] = structure["n_atoms"]
            result["periodic"] = structure["periodic"]
            if not structure["periodic"] or not all(structure["pbc"]):
                result["stress"] = None
        return {
            "model": {"id": MODEL_ID, "revision": MODEL_REVISION, "key": MODEL_KEY, "adapted": self.adapter is not None},
            "units": {"energy": "eV", "forces": "eV/Å", "stress": "eV/Å³"},
            "results": results,
            "seconds": round(time.perf_counter() - started, 3),
        }

    def evaluate(self, structures: Sequence[Mapping[str, Any]], *, batch_size: int = 8) -> dict[str, Any]:
        """Energy and force errors against the reference labels carried by the structures."""
        pass  # standalone rewrite (build_notebook.py): `from .metrics import regression_metrics` removed — names are kernel globals defined by the carried modules

        checked = _check_structures(structures)
        if any(s["energy"] is None or s["forces"] is None for s in checked):
            raise ValueError("every structure needs energy and forces for evaluation")
        prediction = self.predict(checked, batch_size=batch_size)
        return regression_metrics(checked, prediction["results"])

    # ---- adaptation -----------------------------------------------------------------------------------

    def calibrate_e0(self, structures: Sequence[Mapping[str, Any]]) -> dict[str, Any]:
        """Least-squares per-element energy offsets between the reference labels and the current model,
        added to the model's atomic reference energies. Two-parameter change on a Cu/Al dataset; the
        cheapest possible adaptation to a different level of theory. Returns the shifts in eV."""
        import numpy as np
        import torch

        checked = _check_structures(structures)
        if any(s["energy"] is None for s in checked):
            raise ValueError("every structure needs an energy for E0 calibration")
        prediction = self.predict(checked)
        elements = sorted({s for c in checked for s in c["symbols"]})
        counts = np.array([[c["symbols"].count(el) for el in elements] for c in checked], dtype=float)
        residual = np.array([c["energy"] - r["energy"] for c, r in zip(checked, prediction["results"], strict=True)])
        shifts, *_ = np.linalg.lstsq(counts, residual, rcond=None)
        table = _atomic_symbols()
        with torch.no_grad():
            energies = self.model.atomic_energies_fn.atomic_energies
            for element, shift in zip(elements, shifts, strict=True):
                energies[self._z_table.z_to_index(table[element])] += float(shift)
        return {"elements": elements, "shifts_ev": [float(s) for s in shifts], "n_structures": len(checked)}

    def adapt(
        self,
        train: Sequence[Mapping[str, Any]],
        val: Sequence[Mapping[str, Any]] | None = None,
        *,
        epochs: int = 6,
        lr: float = 1e-3,
        batch_size: int = 4,
        trainable_blocks: int = 1,
        energy_weight: float = ENERGY_WEIGHT,
        forces_weight: float = FORCES_WEIGHT,
        calibrate: bool = True,
        seed: int = 0,
        progress: Callable[[dict[str, Any]], None] | None = None,
    ) -> dict[str, Any]:
        """Bounded fine-tuning to the reference energies and forces of `train`.

        Always trains the readouts, the per-element reference energies and the scale/shift block;
        `trainable_blocks` (0..2) additionally unfreezes that many trailing interaction + product
        blocks (1 = the last block, 1.92 M of 8.22 M parameters). Loss = energy_weight × MSE(per-atom
        energy) + forces_weight × MSE(force components), Adam, fixed learning rate, no scheduler. The
        epoch with the lowest validation loss is kept; epoch 0 records the (E0-calibrated) frozen model
        so every number is comparable to the starting point."""
        pass  # standalone rewrite (build_notebook.py): `from .samples import validate_dataset` removed — names are kernel globals defined by the carried modules

        if not isinstance(epochs, int) or not 1 <= epochs <= 50:
            raise ValueError("epochs must be an int in 1..50")
        if not (0.0 < lr <= 0.1):
            raise ValueError("lr must be in (0, 0.1]")
        if not isinstance(batch_size, int) or not 1 <= batch_size <= MAX_STRUCTURES_PER_CALL:
            raise ValueError(f"batch_size must be an int in 1..{MAX_STRUCTURES_PER_CALL}")
        prefixes = _trainable_prefixes(trainable_blocks)
        train_checked = validate_dataset(train, min_records=4)["records"]
        val_checked = validate_dataset(val, min_records=1)["records"] if val else []
        import torch

        torch.manual_seed(seed)
        started = time.perf_counter()
        model = self.model
        pre_state = copy.deepcopy(model.state_dict())  # restored if anything below raises
        calibration = None
        for name, param in model.named_parameters():
            param.requires_grad_(name.startswith(prefixes))
        params = [p for p in model.parameters() if p.requires_grad]
        n_trainable = sum(p.numel() for p in params)
        n_total = sum(p.numel() for p in model.parameters())
        optimiser = torch.optim.Adam(params, lr=lr)
        train_data = self._dataset(train_checked, labels=True)
        val_data = self._dataset(val_checked, labels=True) if val_checked else []

        def weighted_loss(out: Mapping[str, Any], batch: Any) -> Any:
            n_atoms = batch.ptr.diff().to(out["energy"].dtype)
            e_loss = torch.mean(((out["energy"] - batch.energy) / n_atoms) ** 2)
            f_loss = torch.mean((out["forces"] - batch.forces) ** 2)
            return energy_weight * e_loss + forces_weight * f_loss

        def val_loss() -> float | None:
            if not val_data:
                return None
            model.eval()
            total, count = 0.0, 0
            for batch in self._loader(val_data, batch_size, shuffle=False):
                out = self._forward(model, batch, training=False, stress=False)
                total += float(weighted_loss(out, batch)) * batch.num_graphs
                count += batch.num_graphs
            return total / count

        try:
            calibration = self.calibrate_e0(train_checked) if calibrate else None
            history: list[dict[str, Any]] = []
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = 0
            entry: dict[str, Any] = {
                "epoch": 0,
                "train_loss": None,
                "val_loss": val_loss(),
                "note": "frozen model after E0 calibration" if calibrate else "frozen model",
            }
            if val_checked:
                entry["val"] = self.evaluate(val_checked)
            history.append(entry)
            best_val = entry["val_loss"] if entry["val_loss"] is not None else math.inf
            if progress:
                progress(entry)
            for epoch in range(1, epochs + 1):
                model.train()
                total, count = 0.0, 0
                for batch in self._loader(train_data, batch_size, shuffle=True, seed=seed + epoch):
                    out = self._forward(model, batch, training=True, stress=False)
                    loss = weighted_loss(out, batch)
                    optimiser.zero_grad(set_to_none=True)
                    loss.backward()
                    optimiser.step()
                    total += float(loss.detach()) * batch.num_graphs
                    count += batch.num_graphs
                model.eval()
                entry = {"epoch": epoch, "train_loss": total / count, "val_loss": val_loss()}
                if val_checked:
                    entry["val"] = self.evaluate(val_checked)
                history.append(entry)
                if progress:
                    progress(entry)
                if entry["val_loss"] is None or entry["val_loss"] < best_val:
                    best_val = entry["val_loss"] if entry["val_loss"] is not None else best_val
                    best_state = copy.deepcopy(model.state_dict())
                    best_epoch = epoch
        except BaseException:
            # Transactional: a failure in calibration, training, validation or the progress callback
            # leaves the model exactly as it was before adapt(), frozen, with no adapter attached.
            model.load_state_dict(pre_state, strict=True)
            model.eval()
            for param in model.parameters():
                param.requires_grad_(False)
            self.adapter = None
            raise
        model.load_state_dict(best_state, strict=True)
        model.eval()
        for param in model.parameters():
            param.requires_grad_(False)
        self.adapter = {
            "trainable_prefixes": list(prefixes),
            "trainable_blocks": trainable_blocks,
            "n_trainable": n_trainable,
            "n_total": n_total,
            "epochs": epochs,
            "best_epoch": best_epoch,
            "lr": lr,
            "batch_size": batch_size,
            "energy_weight": energy_weight,
            "forces_weight": forces_weight,
            "calibration": calibration,
            "n_train": len(train_checked),
            "n_val": len(val_checked),
            "seed": seed,
            "history": history,
            "seconds": round(time.perf_counter() - started, 2),
        }
        return dict(self.adapter)

    # ---- artifacts ------------------------------------------------------------------------------------

    def save_artifact(self, output_dir: str | Path, metadata: Mapping[str, Any] | None = None) -> Path:
        """Write the adapted tensors (only those the adaptation could change) as safetensors plus a manifest."""
        if self.adapter is None:
            raise ValueError("nothing to save: call adapt() first")
        from safetensors.torch import save_file

        out = Path(output_dir)
        out.mkdir(parents=True, exist_ok=True)
        prefixes = tuple(self.adapter["trainable_prefixes"])
        tensors = {k: v.detach().cpu().contiguous() for k, v in self.model.state_dict().items() if k.startswith(prefixes)}
        weights_path = out / ARTIFACT_WEIGHTS_NAME
        save_file(tensors, str(weights_path), metadata={"format": "pt"})
        manifest = {
            "format": ARTIFACT_FORMAT,
            "format_version": ARTIFACT_FORMAT_VERSION,
            "base_model": {
                "id": MODEL_ID,
                "revision": MODEL_REVISION,
                "key": MODEL_KEY,
                "converted_sha256": dict(CONVERTED_SHA256),
            },
            "adapter": {k: v for k, v in self.adapter.items() if k != "history"},
            "history": self.adapter["history"],
            "tensors": sorted(tensors),
            "files": [
                {"path": ARTIFACT_WEIGHTS_NAME, "bytes": weights_path.stat().st_size, "sha256": _sha256_file(weights_path)}
            ],
            "metadata": dict(metadata or {}),
        }
        (out / ARTIFACT_MANIFEST_NAME).write_text(json.dumps(manifest, indent=2), encoding="utf-8")
        return out

    def _check_artifact_manifest(self, root: Path, manifest: Mapping[str, Any]) -> tuple[Path, list[str]]:
        """Validate an adapter manifest before anything is deserialised: format and version, the pinned base,
        exactly one weights entry named `adapter.safetensors` inside the artifact directory, an in-range
        `trainable_blocks`, and a tensor list equal to the exact set that scope implies for this base."""
        if manifest.get("format") != ARTIFACT_FORMAT:
            raise ValueError(f"artifact format {manifest.get('format')!r} != {ARTIFACT_FORMAT!r}")
        if manifest.get("format_version") != ARTIFACT_FORMAT_VERSION:
            raise ValueError(
                f"artifact format_version {manifest.get('format_version')!r} is not supported "
                f"(expected {ARTIFACT_FORMAT_VERSION!r})"
            )
        base = manifest.get("base_model", {})
        if (base.get("id"), base.get("revision")) != (MODEL_ID, MODEL_REVISION):
            raise ValueError("artifact was adapted from a different base model or revision")
        if base.get("converted_sha256") != CONVERTED_SHA256:
            raise ValueError("artifact records different converted-base digests")
        files = manifest.get("files")
        if not isinstance(files, list) or len(files) != 1:
            raise ValueError("artifact manifest must list exactly one weights file")
        entry = files[0]
        if not isinstance(entry, Mapping) or entry.get("path") != ARTIFACT_WEIGHTS_NAME:
            raise ValueError(f"artifact weights file must be named {ARTIFACT_WEIGHTS_NAME!r}")
        weights_path = (root / entry["path"]).resolve()
        if weights_path.parent != root.resolve():
            raise ValueError("artifact weights file must sit inside the artifact directory")
        adapter = manifest.get("adapter")
        if not isinstance(adapter, Mapping):
            raise ValueError("artifact manifest has no adapter record")
        trainable_blocks = adapter.get("trainable_blocks")
        if isinstance(trainable_blocks, bool) or not isinstance(trainable_blocks, int):
            raise ValueError("artifact adapter.trainable_blocks must be an int")
        prefixes = _trainable_prefixes(trainable_blocks)  # range-checked there
        expected = sorted(k for k in self.model.state_dict() if k.startswith(prefixes))
        if not isinstance(manifest.get("tensors"), list) or sorted(manifest["tensors"]) != expected:
            raise ValueError(
                f"artifact tensor list does not match the {len(expected)} tensors that "
                f"trainable_blocks={trainable_blocks} may change on this base"
            )
        return weights_path, expected

    def load_artifact(self, artifact_dir: str | Path) -> dict[str, Any]:
        """Verify an adapter's manifest, scope and digest, then overwrite exactly the tensors the scope allows."""
        root = Path(artifact_dir)
        manifest = json.loads((root / ARTIFACT_MANIFEST_NAME).read_text(encoding="utf-8"))
        weights_path, expected = self._check_artifact_manifest(root, manifest)
        entry = manifest["files"][0]
        digest = _sha256_file(weights_path)
        if digest != entry["sha256"] or weights_path.stat().st_size != entry["bytes"]:
            raise ValueError(f"{entry['path']}: digest or size mismatch; refusing to load")
        from safetensors.torch import load_file

        tensors = load_file(str(weights_path))
        if sorted(tensors) != expected:
            raise ValueError("artifact tensor names differ from the validated manifest")
        state = self.model.state_dict()
        for key, value in tensors.items():
            if tuple(value.shape) != tuple(state[key].shape):
                raise ValueError(f"artifact tensor {key} has shape {tuple(value.shape)}, base has {tuple(state[key].shape)}")
        merged = dict(state)
        merged.update({k: v.to(state[k].dtype) for k, v in tensors.items()})
        self.model.load_state_dict(merged, strict=True)
        self.model.eval()
        self.adapter = {**manifest["adapter"], "history": manifest.get("history", [])}
        return manifest

    @classmethod
    def from_artifact(
        cls,
        artifact_dir: str | Path,
        *,
        device: str = "cpu",
        weights_dir: str | Path | None = None,
        allow_download: bool = False,
        require_source: bool = True,
    ) -> MaceMaterialsPipeline:
        pipeline = cls.from_pretrained(
            device=device, weights_dir=weights_dir, allow_download=allow_download, require_source=require_source
        )
        pipeline.load_artifact(artifact_dir)
        return pipeline

**Module 3/3:** `src/mace_materials_pipeline/samples.py` (carried verbatim; see the note above)

In [ ]:
"""Tutorial dataset, dataset validation, stratified split and extended-XYZ I/O for the MACE pipeline.

The default dataset is generated in code (no download): rattled and strained face-centred-cubic
2×2×2 supercells of copper, aluminium and a random Cu₁₆Al₁₆ substitutional alloy, **labelled with
ASE's effective-medium-theory potential (EMT)**. EMT is a real, deterministic, analytic interatomic
potential for these metals — not a DFT code and not a surrogate of MACE — so the labels are a
different level of theory from the PBE data MACE-MP-0 was trained on. That is exactly the situation a
foundation potential is fine-tuned in: the frozen model already predicts sensible forces, but its
energies sit on a different reference and its curvature differs, and the adaptation has to close the
gap on held-out structures.

Design choices made so the trivial baselines sit where they should:

* Every structure of one composition has the **same atom count and the same element multiset**, so a
  per-element energy regression (the composition baseline) predicts one energy per composition and
  cannot see the rattle or the strain at all.
* Displacement amplitudes and strains are drawn independently of composition, so nothing about a
  structure's label is recoverable from which elements it contains.
* Structures are labelled with forces as well as energies, so the force error — where the foundation
  model already starts well and must improve further — is measured alongside the energy error.
"""

from __future__ import annotations

import hashlib
import json
import random
from collections.abc import Mapping, Sequence
from pathlib import Path
from typing import Any

# standalone rewrite (build_notebook.py): `from .pipeline import MAX_ATOMS_PER_STRUCTURE, _check_structure, from_atoms, structure_digest, to_atoms` removed — names are kernel globals defined by the carried modules

SAMPLE_COMPOSITIONS = ("Cu32", "Al32", "Cu16Al16")
SAMPLE_SIZE = 48
LATTICE_CONSTANTS = {"Cu": 3.60, "Al": 4.05}  # Å, fcc conventional cell
SUPERCELL = (2, 2, 2)  # 32 atoms
STRAIN_RANGE = 0.03  # isotropic, ±3 %
RATTLE_SIGMAS = (0.05, 0.10, 0.15)  # Å, Gaussian displacement per Cartesian component
SAMPLE_LABEL_SOURCE = "ase.calculators.emt.EMT"
MIN_RECORDS = 8


def _emt_energy_forces(atoms: Any) -> tuple[float, list[list[float]]]:
    from ase.calculators.emt import EMT

    atoms = atoms.copy()
    atoms.calc = EMT()
    energy = float(atoms.get_potential_energy())
    forces = atoms.get_forces().tolist()
    atoms.calc = None
    return energy, forces


def build_sample_structure(composition: str, *, strain: float, sigma: float, rng: random.Random) -> dict[str, Any]:
    """One rattled, strained fcc 2×2×2 supercell of the named composition (geometry only, no labels)."""
    import numpy as np
    from ase.build import bulk

    if composition not in SAMPLE_COMPOSITIONS:
        raise ValueError(f"composition must be one of {SAMPLE_COMPOSITIONS}")
    if composition == "Cu16Al16":
        a0 = 0.5 * (LATTICE_CONSTANTS["Cu"] + LATTICE_CONSTANTS["Al"])  # Vegard's law
        atoms = bulk("Cu", "fcc", a=a0, cubic=True) * SUPERCELL
        symbols = ["Cu"] * len(atoms)
        for index in rng.sample(range(len(atoms)), 16):
            symbols[index] = "Al"
        atoms.set_chemical_symbols(symbols)
    else:
        element = composition[:2]
        atoms = bulk(element, "fcc", a=LATTICE_CONSTANTS[element], cubic=True) * SUPERCELL
    atoms.set_cell(atoms.cell * (1.0 + strain), scale_atoms=True)
    displacement = np.array([[rng.gauss(0.0, sigma) for _ in range(3)] for _ in range(len(atoms))])
    atoms.positions = atoms.positions + displacement
    structure = from_atoms(atoms)
    structure["name"] = f"{composition}-s{strain:+.4f}-r{sigma:.2f}"
    structure["composition"] = composition
    structure["strain"] = strain
    structure["rattle_sigma"] = sigma
    return structure


def generate_sample_dataset(*, seed: int = 42, size: int = SAMPLE_SIZE) -> list[dict[str, Any]]:
    """Deterministic EMT-labelled dataset: `size` structures cycling through the three compositions."""
    if not isinstance(size, int) or not 3 <= size <= 96 or size % 3:
        raise ValueError("size must be a multiple of 3 in 3..96")
    rng = random.Random(seed)
    records = []
    for index in range(size):
        composition = SAMPLE_COMPOSITIONS[index % len(SAMPLE_COMPOSITIONS)]
        strain = rng.uniform(-STRAIN_RANGE, STRAIN_RANGE)
        sigma = RATTLE_SIGMAS[rng.randrange(len(RATTLE_SIGMAS))]
        structure = build_sample_structure(composition, strain=strain, sigma=sigma, rng=rng)
        structure["name"] = f"{structure['name']}-{index:02d}"
        energy, forces = _emt_energy_forces(to_atoms(structure))
        structure["energy"] = energy
        structure["forces"] = forces
        structure["label_source"] = SAMPLE_LABEL_SOURCE
        records.append(structure)
    return records


def composition_key(structure: Mapping[str, Any]) -> str:
    """Hill-order formula string, e.g. Al16Cu16, used to stratify splits."""
    counts: dict[str, int] = {}
    for symbol in structure["symbols"]:
        counts[symbol] = counts.get(symbol, 0) + 1
    return "".join(f"{el}{counts[el]}" for el in sorted(counts))


def validate_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    min_records: int = MIN_RECORDS,
    require_labels: bool = True,
) -> dict[str, Any]:
    """Structural validation of a labelled dataset; raises ValueError before any model import."""
    if isinstance(records, Mapping) or not isinstance(records, Sequence) or isinstance(records, (str, bytes)):
        raise ValueError("records must be a list of structure mappings")
    if len(records) < min_records:
        raise ValueError(f"{len(records)} records; at least {min_records} are required")
    checked = []
    names: set[str] = set()
    for index, record in enumerate(records):
        structure = _check_structure(record, index)
        if require_labels and (structure["energy"] is None or structure["forces"] is None):
            raise ValueError(f"{structure['name']}: energy and forces are required")
        if structure["name"] in names:
            raise ValueError(f"duplicate structure name {structure['name']!r}")
        names.add(structure["name"])
        structure["composition"] = composition_key(structure)
        for key in ("strain", "rattle_sigma", "label_source"):
            if key in record:
                structure[key] = record[key]
        checked.append(structure)
    compositions: dict[str, int] = {}
    for structure in checked:
        compositions[structure["composition"]] = compositions.get(structure["composition"], 0) + 1
    return {
        "records": checked,
        "n_records": len(checked),
        "n_atoms": sum(s["n_atoms"] for s in checked),
        "compositions": compositions,
        "elements": sorted({s for c in checked for s in c["symbols"]}),
        "max_atoms": max(s["n_atoms"] for s in checked),
        "labelled": all(s["energy"] is not None and s["forces"] is not None for s in checked),
        "digest": dataset_digest(checked),
    }


def dataset_digest(records: Sequence[Mapping[str, Any]]) -> str:
    """SHA-256 over per-structure geometry digests and labels rounded to 1e-6."""
    parts = []
    for record in records:
        energy = None if record.get("energy") is None else round(float(record["energy"]), 6)
        forces = None if record.get("forces") is None else [[round(float(x), 6) for x in row] for row in record["forces"]]
        parts.append([structure_digest(record), energy, forces])
    return hashlib.sha256(json.dumps(parts, separators=(",", ":")).encode("utf-8")).hexdigest()


def split_dataset(
    records: Sequence[Mapping[str, Any]],
    *,
    val_fraction: float = 0.2,
    test_fraction: float = 0.25,
    seed: int = 0,
) -> dict[str, list[dict[str, Any]]]:
    """Shuffle within each composition and cut val/test fractions so every split sees every composition."""
    if not (0.0 <= val_fraction < 1.0 and 0.0 < test_fraction < 1.0 and val_fraction + test_fraction < 1.0):
        raise ValueError("fractions must satisfy 0 <= val < 1, 0 < test < 1, val + test < 1")
    checked = validate_dataset(records, min_records=4)["records"]
    by_composition: dict[str, list[dict[str, Any]]] = {}
    for structure in checked:
        by_composition.setdefault(structure["composition"], []).append(structure)
    rng = random.Random(seed)
    splits: dict[str, list[dict[str, Any]]] = {"train": [], "val": [], "test": []}
    for composition in sorted(by_composition):
        group = list(by_composition[composition])
        rng.shuffle(group)
        n_test = max(1, round(len(group) * test_fraction))
        n_val = round(len(group) * val_fraction)
        splits["test"].extend(group[:n_test])
        splits["val"].extend(group[n_test : n_test + n_val])
        splits["train"].extend(group[n_test + n_val :])
    if not splits["train"]:
        raise ValueError("split leaves no training structures")
    return splits


def load_byod_dataset(path: str | Path) -> list[dict[str, Any]]:
    """Read an extended-XYZ file (`ase.io.read`, all frames) into structure mappings with labels from
    `energy` / `forces` (info/arrays or the attached single-point results)."""
    import ase.io

    file_path = Path(path)
    if not file_path.is_file():
        raise FileNotFoundError(f"dataset not found: {file_path}")
    if file_path.suffix.lower() not in (".xyz", ".extxyz"):
        raise ValueError("BYOD datasets must be extended XYZ (.xyz / .extxyz)")
    frames = ase.io.read(str(file_path), index=":", format="extxyz")
    if not isinstance(frames, list):
        frames = [frames]
    records = []
    for index, atoms in enumerate(frames):
        if len(atoms) > MAX_ATOMS_PER_STRUCTURE:
            raise ValueError(f"frame {index}: {len(atoms)} atoms exceed {MAX_ATOMS_PER_STRUCTURE}")
        record = from_atoms(atoms, name=atoms.info.get("name") or f"{file_path.stem}-{index:04d}")
        records.append(record)
    return records


def write_dataset_xyz(records: Sequence[Mapping[str, Any]], path: str | Path) -> Path:
    """Write structures (with energy/forces when present) as extended XYZ, the shape BYOD expects."""
    import ase.io
    from ase.calculators.singlepoint import SinglePointCalculator

    frames = []
    for record in records:
        atoms = to_atoms(record)
        atoms.info.pop("energy", None)
        forces = atoms.arrays.pop("forces", None)
        if record.get("energy") is not None and forces is not None:
            atoms.calc = SinglePointCalculator(atoms, energy=float(record["energy"]), forces=forces)
        frames.append(atoms)
    out = Path(path)
    out.parent.mkdir(parents=True, exist_ok=True)
    ase.io.write(str(out), frames, format="extxyz")
    return out

## 3. Pin, stage and verify the model

The model identity is carried twice — `MODEL_ID`/`MODEL_REVISION` in the module above and the `2`-file manifest below (paths, byte sizes, SHA-256) — and the cell first asserts they agree. It writes the manifest into the working-directory snapshot, then `stage_missing_files(..., allow_download=True)` fetches exactly the entries that are absent from the Hugging Face Hub **at revision `e291ace2bfae…`** (never `main`), `verify_snapshot` re-hashes every file and raises on the first size or digest mismatch, and only then does `MaceMaterialsPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, device=('cuda' if torch.cuda.is_available() else 'cpu'), report=print)` load the verified files. There is no fallback to a different download and no remote model code is executed. The effective identity, device and weight source are printed before any inference.

In [ ]:
import json

MANIFEST = {
  "format": "dimer_hf_snapshot",
  "formatVersion": 1,
  "modelKey": "mace-mp-0b2-small",
  "modelId": "mace-foundations/mace-mp-0",
  "revision": "e291ace2bfae073c3ebc7ae2f9479a525989baa7",
  "files": [
    {
      "path": "README.md",
      "bytes": 24,
      "sha256": "d8d7a46d41a1a37fe4f0a5f637bf55c649310185329127d8a2204632e480be17"
    },
    {
      "path": "mace-mp-0b2-small.model",
      "bytes": 67622684,
      "sha256": "d5773bf9440e96d6eb8c598f84bd0e6369fcfa432f626a87f890e07da3c651c9",
      "note": "Pickled torch module (executable serialization, asset spec 11.2). Byte-identical to the upstream GitHub release asset ACEsuit/mace-mp mace_mp_0b2/mace-small-density-agnesi-stress.model. Never served: statically audited and converted once into mace-mp-0b2-small.config.json + mace-mp-0b2-small.safetensors (digests pinned in pipeline.py)."
    }
  ],
  "totalBytes": 67622708
}

if (MANIFEST['modelId'], MANIFEST['revision']) != (MODEL_ID, MODEL_REVISION):
    raise RuntimeError('inline manifest does not name the identity carried by the pipeline module; the notebook was not regenerated after a change')
WEIGHTS_DIR = DEFAULT_WEIGHTS_DIR
WEIGHTS_DIR.mkdir(parents=True, exist_ok=True)
with open(WEIGHTS_DIR / MANIFEST_NAME, 'w', encoding='utf-8') as handle:
    json.dump(MANIFEST, handle, indent=2)
print({'model_id': MODEL_ID, 'revision': MODEL_REVISION, 'license': MODEL_LICENSE, 'files': len(MANIFEST['files']), 'total_bytes': MANIFEST['totalBytes']})
fetched = stage_missing_files(WEIGHTS_DIR, allow_download=True)
print({'weights_dir': str(WEIGHTS_DIR), 'fetched': fetched})
snapshot = verify_snapshot(WEIGHTS_DIR)
_files = snapshot.get('files', []) if isinstance(snapshot, dict) else []
print({'verified_files': len(_files) if isinstance(_files, list) else _files, 'revision': snapshot.get('revision', MODEL_REVISION) if isinstance(snapshot, dict) else MODEL_REVISION})
pipe = MaceMaterialsPipeline.from_pretrained(weights_dir=WEIGHTS_DIR, device=('cuda' if torch.cuda.is_available() else 'cpu'), report=print)
print({'device': getattr(pipe, 'device', None), 'source': getattr(pipe, 'source', 'local-snapshot')})

## 4. Sample dataset, validation and split

The default dataset is generated in code with a fixed seed: 16 structures each of Cu₃₂, Al₃₂ and a random substitutional Cu₁₆Al₁₆ alloy — 2×2×2 fcc supercells with an isotropic strain drawn from ±3 % and Gaussian rattles of σ = 0.05, 0.10 or 0.15 Å — labelled with energies and forces from ASE's EMT calculator. `validate_dataset` checks every structure (element support, distances, cell/pbc, label shapes) and the dataset contract before any model runs; `split_dataset` shuffles within each composition and cuts 20 % validation / 25 % test, so every split sees every composition.

Look for: 48 structures, three compositions of 16, 1,536 atoms, splits 27/9/12, EMT energies of a few tenths of an eV per atom, and a written `outputs/mace_materials_sample_dataset.xyz` in the extended-XYZ shape BYOD expects. Four refusal probes follow — an unsupported element, overlapping atoms, periodicity without a cell, and a malformed force array — each rejected before `torch` does anything.

In [ ]:
import json
import os
from pathlib import Path

USE_BYOD = False  # @param {type:"boolean"}
VAL_FRACTION = 0.2  # @param {type:"number"}
TEST_FRACTION = 0.25  # @param {type:"number"}
SEED = 42  # @param {type:"integer"}

os.makedirs('outputs', exist_ok=True)
if USE_BYOD:
    from google.colab import files
    uploaded = files.upload()
    file_name, payload = next(iter(uploaded.items()))
    byod_path = Path('work') / file_name
    byod_path.parent.mkdir(parents=True, exist_ok=True)
    byod_path.write_bytes(payload)
    records = load_byod_dataset(byod_path)
    data_source = 'BYOD (' + file_name + ')'
else:
    records = generate_sample_dataset(seed=SEED)
    data_source = f'generated fcc supercells labelled by {SAMPLE_LABEL_SOURCE} (seed {SEED}, {SAMPLE_SIZE} structures)'

dataset_manifest = validate_dataset(records)
splits = split_dataset(records, val_fraction=VAL_FRACTION, test_fraction=TEST_FRACTION, seed=SEED)
train_records, val_records, test_records = splits['train'], splits['val'], splits['test']
write_dataset_xyz(records, 'outputs/mace_materials_sample_dataset.xyz')

print({'data_source': data_source, 'n_records': dataset_manifest['n_records'], 'n_atoms': dataset_manifest['n_atoms'], 'compositions': dataset_manifest['compositions']})
print({'elements': dataset_manifest['elements'], 'max_atoms': dataset_manifest['max_atoms'], 'labelled': dataset_manifest['labelled'], 'digest': dataset_manifest['digest'][:16] + '...'})
print({'energy_per_atom_range_eV': (round(min(r['energy'] / len(r['symbols']) for r in records), 4), round(max(r['energy'] / len(r['symbols']) for r in records), 4))})
print({'train': len(train_records), 'validation': len(val_records), 'test': len(test_records)})
print({'example': {k: (v if not isinstance(v, list) else f'list[{len(v)}]') for k, v in records[0].items()}})

print({'validation': INPUT_SCHEMA['validation']})
probes = {
    'unsupported element': {'symbols': ['Po', 'Cu'], 'positions': [[0, 0, 0], [2.5, 0, 0]]},
    'overlapping atoms': {'symbols': ['Cu', 'Cu'], 'positions': [[0, 0, 0], [0.3, 0, 0]]},
    'pbc without a cell': {'symbols': ['Cu'], 'positions': [[0, 0, 0]], 'pbc': True},
    'malformed forces': {**records[0], 'forces': [[0.0, 0.0]] * len(records[0]['symbols'])},
}
for name, structure in probes.items():
    try:
        validate_inputs([structure])
        print({'probe': name, 'verdict': 'accepted'})
    except (TypeError, ValueError) as exc:
        print({'probe': name, 'rejected': str(exc)[:110]})

## 5. Zero-shot prediction and physics checks on the frozen model

`pipe.predict` returns, per structure, the total energy (eV), the per-atom energy, per-atom energy contributions, forces (eV/Å) and — for fully periodic cells — the stress tensor (eV/Å³). The model is the pinned foundation potential, untouched.

Five checks say whether the rebuilt model behaves like an interatomic potential should. **Rotation:** rotating the cell and every atom leaves the energy unchanged and rotates the forces with it (equivariance). **Translation** and **permutation** of atoms change nothing. **Gradient consistency:** the force on one atom equals the central finite difference of the energy along that coordinate, to ~1e-8 eV/Å — the forces are the analytic gradient, not a separate head. **Extensivity:** a 2×1×1 supercell has twice the energy. Look for differences at the 1e-13 level for the symmetry checks and 1e-8 for the finite difference; a rebuilt model with a wrong config would fail these before any accuracy number.

In [ ]:
import time

import numpy as np

s0 = test_records[0]
t0 = time.perf_counter()
prediction = pipe.predict(test_records[:8])
print({'structures': 8, 'seconds': round(time.perf_counter() - t0, 2), 'units': prediction['units']})
base = prediction['results'][0]
print({'name': base['name'], 'energy_eV': round(base['energy'], 4), 'energy_per_atom_eV': round(base['energy_per_atom'], 4), 'reference_eV': round(s0['energy'], 4), 'max_force_eV_A': round(float(np.abs(base['forces']).max()), 4), 'stress_diag_eV_A3': [round(base['stress'][i][i], 5) for i in range(3)]})

rng = np.random.default_rng(0)
q, _ = np.linalg.qr(rng.normal(size=(3, 3)))
if np.linalg.det(q) < 0:
    q[:, 0] *= -1
rotated = {**s0, 'positions': (np.array(s0['positions']) @ q.T).tolist(), 'cell': (np.array(s0['cell']) @ q.T).tolist()}
rot = pipe.predict([rotated])['results'][0]
translated = {**s0, 'positions': (np.array(s0['positions']) + 1.234).tolist()}
tr = pipe.predict([translated])['results'][0]
perm = rng.permutation(len(s0['symbols']))
permuted = {**s0, 'symbols': [s0['symbols'][i] for i in perm], 'positions': [s0['positions'][i] for i in perm]}
pm = pipe.predict([permuted])['results'][0]
h = 1e-4
plus, minus = np.array(s0['positions']), np.array(s0['positions'])
plus[3, 0] += h
minus[3, 0] -= h
e_plus = pipe.predict([{**s0, 'positions': plus.tolist()}])['results'][0]['energy']
e_minus = pipe.predict([{**s0, 'positions': minus.tolist()}])['results'][0]['energy']
fd_force = -(e_plus - e_minus) / (2 * h)
supercell = pipe.predict([from_atoms(to_atoms(s0) * (2, 1, 1))])['results'][0]
physics = {
    'rotation_energy_diff': abs(rot['energy'] - base['energy']),
    'rotation_force_diff': float(np.abs(np.array(rot['forces']) - np.array(base['forces']) @ q.T).max()),
    'translation_energy_diff': abs(tr['energy'] - base['energy']),
    'permutation_energy_diff': abs(pm['energy'] - base['energy']),
    'permutation_force_diff': float(np.abs(np.array(pm['forces']) - np.array(base['forces'])[perm]).max()),
    'finite_difference_force': fd_force,
    'analytic_force': base['forces'][3][0],
    'gradient_consistency_diff': abs(fd_force - base['forces'][3][0]),
    'extensivity_diff_per_atom': abs(supercell['energy'] - 2 * base['energy']) / supercell['n_atoms'],
    'batch_vs_single_energy_diff': abs(pipe.predict([s0])['results'][0]['energy'] - base['energy']),
}
for key, value in physics.items():
    print({key: f'{value:.3e}'})
assert physics['rotation_energy_diff'] < 1e-8 and physics['rotation_force_diff'] < 1e-8
assert physics['permutation_energy_diff'] < 1e-8 and physics['translation_energy_diff'] < 1e-8
assert physics['gradient_consistency_diff'] < 1e-6
assert physics['extensivity_diff_per_atom'] < 1e-8

## 6. Baselines and the zero-shot error

Three numbers frame everything that follows. The **composition baseline** fits one energy per element on the training split (the classical E0 regression) and predicts zero forces: it is blind to geometry by construction, so its energy error is the within-composition spread of the labels and its force error is the mean absolute reference force. The **zero-force baseline** is that force number alone. The **frozen foundation model** is evaluated as is: expect a force MAE well below the zero baseline — MACE-MP-0 already knows how metals push on each other — but an energy error of several eV per atom, because PBE total energies and EMT energies sit on different absolute references. That offset is the first thing adaptation removes.

Energies are per atom (eV/atom); forces are per Cartesian component (eV/Å).

In [ ]:
baseline_composition = composition_baseline(train_records, test_records)
baseline_zero_force = zero_force_baseline(test_records)
t0 = time.perf_counter()
zero_shot_test = pipe.evaluate(test_records)
zero_shot_seconds = round(time.perf_counter() - t0, 2)
print({'composition_baseline': {'energy_mae_per_atom': round(baseline_composition['energy_mae_per_atom'], 4), 'force_mae': round(baseline_composition['force_mae'], 4), 'e0_eV': {k: round(v, 4) for k, v in baseline_composition['e0_ev'].items()}}})
print({'zero_force_baseline': {'force_mae': round(baseline_zero_force['force_mae'], 4), 'force_rmse': round(baseline_zero_force['force_rmse'], 4)}})
print({'frozen_foundation_model': {k: round(v, 4) for k, v in zero_shot_test.items() if isinstance(v, float)}, 'seconds': zero_shot_seconds})
assert zero_shot_test['force_mae'] < baseline_zero_force['force_mae']

## 7. Calibrate and fine-tune

`pipe.adapt` does two things in order. First it **calibrates the per-element reference energies**: a least-squares fit of one offset per element between the training labels and the frozen predictions, added to the model's atomic energies — two numbers on a Cu/Al dataset, and the cheapest possible move to a new level of theory. That calibrated frozen model is recorded as epoch 0 so every later number is comparable to it. Then it trains the readouts, the reference energies, the scale/shift block and — with `TRAINABLE_BLOCKS = 1` — the last interaction and product blocks: 1.92 M of 8.22 M parameters, Adam at a fixed learning rate, loss = 100 × MSE(per-atom energy) + 10 × MSE(force components), batches of 4, no scheduler, seeded shuffling. The epoch with the lowest validation loss is kept.

Watch the validation energy MAE fall from ~0.1 eV/atom (the calibrated frozen model) to below 0.01 and the force MAE roughly halve over eight epochs, about a minute on CPU. The counter-examples are worth running once: `TRAINABLE_BLOCKS = 0` (readouts only, 2,192 parameters) fixes energies slowly and barely moves forces; `TRAINABLE_BLOCKS = 2` trains everything and is slower without being better on this small set.

In [ ]:
EPOCHS = 8  # @param {type:"integer"}
LEARNING_RATE = 1e-3  # @param {type:"number"}
BATCH_SIZE = 4  # @param {type:"integer"}
TRAINABLE_BLOCKS = 1  # @param {type:"integer"}

def report(entry):
    row = {'epoch': entry['epoch'], 'train_loss': None if entry['train_loss'] is None else round(entry['train_loss'], 4), 'val_loss': round(entry['val_loss'], 4)}
    if 'val' in entry:
        row['val_energy_mae_per_atom'] = round(entry['val']['energy_mae_per_atom'], 5)
        row['val_force_mae'] = round(entry['val']['force_mae'], 5)
    if 'note' in entry:
        row['note'] = entry['note']
    print(row)

t0 = time.perf_counter()
adapt_result = pipe.adapt(train_records, val_records, epochs=EPOCHS, lr=LEARNING_RATE, batch_size=BATCH_SIZE, trainable_blocks=TRAINABLE_BLOCKS, progress=report)
adapt_seconds = round(time.perf_counter() - t0, 1)
print({'trainable_parameters': adapt_result['n_trainable'], 'total_parameters': adapt_result['n_total'], 'best_epoch': adapt_result['best_epoch'], 'seconds': adapt_seconds})
print({'e0_calibration_eV': dict(zip(adapt_result['calibration']['elements'], [round(s, 4) for s in adapt_result['calibration']['shifts_ev']]))})
print({'trainable_prefixes': adapt_result['trainable_prefixes']})

## 8. Held-out evaluation

The test split was never used for calibration, training or epoch selection. The adapted numbers are read against the composition baseline (what a model that cannot see geometry achieves), the zero-force baseline and the frozen foundation model from Section 6. Look for an energy MAE an order of magnitude below the composition baseline and a force MAE well below the frozen model's. Twelve test structures from one seeded split give no dispersion estimate — the deltas are sample-sanity evidence that the adaptation contract works, not a benchmark.

In [ ]:
test_metrics = pipe.evaluate(test_records)
val_metrics = pipe.evaluate(val_records)
print({'test_adapted': {k: round(v, 5) for k, v in test_metrics.items() if isinstance(v, float)}, 'n_structures': test_metrics['n_structures']})
comparison = {
    'energy_mae_per_atom': {'composition_baseline': round(baseline_composition['energy_mae_per_atom'], 5), 'frozen_model': round(zero_shot_test['energy_mae_per_atom'], 5), 'adapted': round(test_metrics['energy_mae_per_atom'], 5)},
    'force_mae': {'zero_force_baseline': round(baseline_zero_force['force_mae'], 5), 'frozen_model': round(zero_shot_test['force_mae'], 5), 'adapted': round(test_metrics['force_mae'], 5)},
}
for metric, values in comparison.items():
    print({metric: values})
evaluation_report = {
    'model': {'id': MODEL_ID, 'revision': MODEL_REVISION, 'key': MODEL_KEY},
    'data_source': data_source,
    'dataset_digest': dataset_manifest['digest'],
    'splits': {'train': len(train_records), 'validation': len(val_records), 'test': len(test_records)},
    'baselines': {'composition': baseline_composition, 'zero_force': baseline_zero_force, 'frozen_model': zero_shot_test},
    'physics_checks': physics,
    'validation_metrics': val_metrics,
    'test_metrics': test_metrics,
    'comparison': comparison,
    'adaptation': {k: v for k, v in adapt_result.items() if k != 'history'},
    'history': adapt_result['history'],
    'adaptation_seconds': adapt_seconds,
}
with open('outputs/mace_materials_evaluation_report.json', 'w', encoding='utf-8') as f:
    json.dump(evaluation_report, f, indent=2)
assert test_metrics['energy_mae_per_atom'] < baseline_composition['energy_mae_per_atom']
assert test_metrics['force_mae'] < zero_shot_test['force_mae']
print({'report': 'outputs/mace_materials_evaluation_report.json'})

## 9. Inference on new structures, artifact export and fresh reload

Three structures are generated with a different seed — one per composition — and labelled with EMT so the adapted model's errors on them can be printed alongside its predictions; they were never part of any split. The stress tensor is returned because the cells are fully periodic.

`pipe.save_artifact` writes only the tensors the adaptation could change (readouts, reference energies, scale/shift and the unfrozen blocks) as `adapter.safetensors`, with a `manifest.json` recording the artifact format, the base model id and revision, the digests of the converted base pair, the tensor names, the file size and SHA-256, the training configuration and the epoch history (OUT8). `MaceMaterialsPipeline.from_artifact` re-verifies the base snapshot, checks the artifact manifest and digest **before** deserialising, and overlays the tensors onto a freshly rebuilt base — a new object from files, not the in-memory model (VER2). The cell asserts identical energies and forces (VER4).

In [ ]:
import random
import shutil

if USE_BYOD:
    new_records = test_records[:3]
    new_source = 'first three BYOD test-split structures'
else:
    new_rng = random.Random(7)
    new_records = []
    for composition in SAMPLE_COMPOSITIONS:
        structure = build_sample_structure(composition, strain=new_rng.uniform(-0.03, 0.03), sigma=0.10, rng=new_rng)
        structure['energy'], structure['forces'] = _emt_energy_forces(to_atoms(structure))
        new_records.append(structure)
    new_source = 'freshly generated fcc supercells (seed 7), one per composition'
new_prediction = pipe.predict(new_records)
print({'new_source': new_source, 'adapted': new_prediction['model']['adapted']})
for record, result in zip(new_records, new_prediction['results']):
    ref_forces = np.array(record['forces'])
    print({'name': result['name'], 'n_atoms': result['n_atoms'], 'energy_per_atom_eV': round(result['energy_per_atom'], 4), 'reference_per_atom_eV': round(record['energy'] / result['n_atoms'], 4), 'force_mae_eV_A': round(float(np.abs(np.array(result['forces']) - ref_forces).mean()), 4), 'pressure_eV_A3': round(-sum(result['stress'][i][i] for i in range(3)) / 3, 5)})
new_metrics = pipe.evaluate(new_records)
print({'new_structures': {k: round(v, 5) for k, v in new_metrics.items() if isinstance(v, float)}, 'note': 'sanity check on generated structures, not an evaluation'})

with open('outputs/mace_materials_predictions.json', 'w', encoding='utf-8') as f:
    json.dump({'source': new_source, 'units': new_prediction['units'], 'results': [{k: v for k, v in r.items() if k != 'node_energies'} for r in new_prediction['results']]}, f, indent=2)

artifact_dir = Path('outputs/mace_materials_adapter')
shutil.rmtree(artifact_dir, ignore_errors=True)
pipe.save_artifact(artifact_dir, metadata={'tutorial': 'mace_materials', 'data_source': data_source})
artifact_manifest = json.loads((artifact_dir / 'manifest.json').read_text(encoding='utf-8'))
print({'artifact': str(artifact_dir), 'format': artifact_manifest['format'], 'tensors': len(artifact_manifest['tensors']), 'bytes': artifact_manifest['files'][0]['bytes'], 'sha256': artifact_manifest['files'][0]['sha256'][:16] + '...'})

reloaded = MaceMaterialsPipeline.from_artifact(artifact_dir, weights_dir=WEIGHTS_DIR, device=pipe.device)
before = pipe.predict(test_records[:4])['results']
after = reloaded.predict(test_records[:4])['results']
parity = {
    'max_abs_energy_diff': max(abs(a['energy'] - b['energy']) for a, b in zip(before, after)),
    'max_abs_force_diff': max(float(np.abs(np.array(a['forces']) - np.array(b['forces'])).max()) for a, b in zip(before, after)),
}
print({'reload_parity': parity, 'reloaded_best_epoch': reloaded.adapter['best_epoch']})
assert parity['max_abs_energy_diff'] < 1e-9 and parity['max_abs_force_diff'] < 1e-9

import platform

import safetensors

result_payload = {
    'notebook_source': NOTEBOOK_SOURCE,
    'repository_revision': NOTEBOOK_SOURCE['repository_revision'],
    'model': {**evaluation_report['model'], 'model_license': MODEL_LICENSE, 'device': pipe.device, 'source': pipe.source},
    'provenance': {
        'source_asset': next(e for e in MANIFEST['files'] if e['path'] == SOURCE_MODEL_NAME),
        'pickle_audit_sha256': PICKLE_AUDIT_SHA256,
        'converted': verify_converted(WEIGHTS_DIR)['files'],
        'pickle_unpickled_once_for_conversion': True,
        'served_from_pickle': False,
        'remote_code_executed': False,
    },
    'runtime': {'python': platform.python_version(), 'torch': torch.__version__, 'mace': mace.__version__, 'e3nn': e3nn.__version__, 'ase': ase.__version__, 'safetensors': safetensors.__version__},
    'data_source': data_source,
    'test_metrics': test_metrics,
    'comparison': comparison,
    'new_structures': new_metrics,
    'artifact': {'dir': str(artifact_dir), 'sha256': artifact_manifest['files'][0]['sha256'], 'bytes': artifact_manifest['files'][0]['bytes']},
    'reload_parity': parity,
}
with open('outputs/mace_materials_result.json', 'w', encoding='utf-8') as f:
    json.dump(result_payload, f, indent=2)

print('outputs/:')
for path in sorted(Path('outputs').rglob('*')):
    if path.is_file():
        print(f'  - {path.as_posix()} ({path.stat().st_size / 1024:.1f} KB)')

## Interpretation and limits

The frozen foundation potential already predicts EMT forces to roughly a fifth of an eV/Å without ever having seen EMT — that is what a universal potential trained on Materials Project trajectories carries into a new system — but its energies sit several eV per atom away, because PBE and EMT put their zero in different places. Two per-element offsets remove that; a bounded fine-tuning of the readouts and the last interaction block then brings the held-out energy error an order of magnitude below the composition baseline and roughly halves the force error, in about a minute on CPU. That is the claim: the adaptation contract moves a foundation interatomic potential onto a different level of theory from a few dozen labelled structures, and the physics checks show the rebuilt, converted model is still an equivariant potential whose forces are the gradient of its energy.

The test split has 12 generated supercells of two elements, the metrics come from one seeded split with no dispersion estimate, and EMT is a toy level of theory chosen because it runs in the notebook. So a small error here says the contract works, not that the adapted model reproduces DFT for Cu–Al, transfers to other compositions, defects or surfaces, or that it is stable in molecular dynamics — none of which this repository exercises. Fine-tuning a foundation potential on a narrow dataset can also erode its behaviour elsewhere; upstream mitigates that with multi-head replay against the original training data, which is out of scope here.

Three things to carry to real data. **Reference consistency:** every label must come from one code, one functional and one set of settings; mixing references is the most common way to get an unlearnable dataset. **Splits:** structures from one trajectory or one relaxation are near-duplicates — split by trajectory or by composition, never at random over frames. **Baselines first:** if the composition baseline is already good, your labels vary with stoichiometry more than with geometry, and the force error is the number to read.

Successful execution proves that the recorded repository revision's pipeline modules, carried in this standalone notebook, can acquire and digest-verify the pinned source asset, audit and convert a pickled checkpoint into a code-free serving pair without executing anything outside the audited allow-lists, rebuild the model from the installed library, validate the demonstrated dataset contract, execute bounded fine-tuning, evaluate against two trivial baselines and the frozen model on an independent split, and emit the shown machine-readable artifacts — without the repository being reachable. It does **not** establish benchmark superiority, production fitness, or physical validity beyond the checks shown.

**Optional experiments (they do not affect the default path):** set `TRAINABLE_BLOCKS = 0` to train the readouts alone and watch the forces stay put; set `TRAINABLE_BLOCKS = 2` to fine-tune everything and compare the time; raise `EPOCHS`; or bring your own extended-XYZ dataset through BYOD and read the composition baseline before the adapted number.

## References

- Repository README: https://github.com/kurtvalcorza/mace-materials-pipeline/blob/main/README.md
- Repository model card: https://github.com/kurtvalcorza/mace-materials-pipeline/blob/main/MODEL_CARD.md
- Weights and conversion notes: https://github.com/kurtvalcorza/mace-materials-pipeline/blob/main/docs/WEIGHTS.md
- Hugging Face model repository: https://huggingface.co/mace-foundations/mace-mp-0 (revision `e291ace2bfae073c3ebc7ae2f9479a525989baa7`)
- Upstream release asset (byte-identical): https://github.com/ACEsuit/mace-mp/releases/tag/mace_mp_0b2
- Batatia et al., *A foundation model for atomistic materials chemistry*, arXiv:2401.00096 (2023): https://arxiv.org/abs/2401.00096
- Batatia et al., *MACE: Higher order equivariant message passing neural networks for fast and accurate force fields*, NeurIPS 2022: https://arxiv.org/abs/2206.07697
- ASE EMT calculator: https://wiki.fysik.dtu.dk/ase/ase/calculators/emt.html
- DIMER Notebook Specification 2.0 and Model Card Specification 1.1 (fleet specs in the ml-worker repository)